### Import libraries and load staging patient data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
STAGING_DATA_PATH = Path("../data/staging")
WAREHOUSE_DATA_PATH = Path("../data/processed/warehouse")

WAREHOUSE_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
stg_patients = pd.read_csv(
    STAGING_DATA_PATH / "stg_patients.csv"
)

print(
    "Staging patient rows:",
    len(stg_patients)
)

stg_patients.head()

Staging patient rows: 9981


,PatientID,BirthDate,Gender,Race,Ethnicity,ZipCode,State,Region,RegistrationDate,BirthDateValidFlag
0,UNKNOWN,NaN,Unknown,Unknown,Unknown,Unknown,UN,Unknown,NaN,0
1,PAT000001,2006-07-24,Male,White,Not Hispanic or Latino,22554,VA,Northern,2023-12-08,1
2,PAT000002,1953-04-07,Female,White,Not Hispanic or Latino,23834,VA,Southern,NaN,1
3,PAT000003,1994-03-09,Male,White,Not Hispanic or Latino,23606,VA,Eastern,NaN,1
4,PAT000004,2023-08-26,Female,Black or African American,Not Hispanic or Latino,22401,VA,Northern,NaN,1


In [4]:
stg_patients = (
    stg_patients
    .assign(
        _UnknownSort=np.where(
            stg_patients["PatientID"] == "UNKNOWN",
            0,
            1
        )
    )
    .sort_values(
        [
            "_UnknownSort",
            "PatientID"
        ]
    )
    .drop(
        columns="_UnknownSort"
    )
    .reset_index(drop=True)
)

In [5]:
stg_patients.head()

,PatientID,BirthDate,Gender,Race,Ethnicity,ZipCode,State,Region,RegistrationDate,BirthDateValidFlag
0,UNKNOWN,NaN,Unknown,Unknown,Unknown,Unknown,UN,Unknown,NaN,0
1,PAT000001,2006-07-24,Male,White,Not Hispanic or Latino,22554,VA,Northern,2023-12-08,1
2,PAT000002,1953-04-07,Female,White,Not Hispanic or Latino,23834,VA,Southern,NaN,1
3,PAT000003,1994-03-09,Male,White,Not Hispanic or Latino,23606,VA,Eastern,NaN,1
4,PAT000004,2023-08-26,Female,Black or African American,Not Hispanic or Latino,22401,VA,Northern,NaN,1


In [6]:
dim_patient = stg_patients.copy()

dim_patient.insert(
    0,
    "PatientKey",
    range(len(dim_patient))
)

In [7]:
dim_patient.head(10)

,PatientKey,PatientID,BirthDate,Gender,Race,Ethnicity,ZipCode,State,Region,RegistrationDate,BirthDateValidFlag
0,0,UNKNOWN,NaN,Unknown,Unknown,Unknown,Unknown,UN,Unknown,NaN,0
1,1,PAT000001,2006-07-24,Male,White,Not Hispanic or Latino,22554,VA,Northern,2023-12-08,1
2,2,PAT000002,1953-04-07,Female,White,Not Hispanic or Latino,23834,VA,Southern,NaN,1
3,3,PAT000003,1994-03-09,Male,White,Not Hispanic or Latino,23606,VA,Eastern,NaN,1
4,4,PAT000004,2023-08-26,Female,Black or African American,Not Hispanic or Latino,22401,VA,Northern,NaN,1
5,5,PAT000005,2007-02-25,Female,White,Hispanic or Latino,23805,VA,Southern,2023-07-12,1
6,6,PAT000006,1994-08-16,Male,White,Hispanic or Latino,22902,VA,Western,2025-01-04,1
7,7,PAT000007,2024-03-13,Female,White,Not Hispanic or Latino,23220,VA,Central,NaN,1
8,8,PAT000008,2017-08-10,Other,Asian,Not Hispanic or Latino,23225,VA,Central,NaN,1
9,9,PAT000009,2021-04-10,Male,Black or African American,Not Hispanic or Latino,23601,VA,Eastern,NaN,1


In [8]:
dim_patient["BirthDate"] = pd.to_datetime(
    dim_patient["BirthDate"],
    format="mixed",
    errors="coerce"
)

dim_patient["RegistrationDate"] = pd.to_datetime(
    dim_patient["RegistrationDate"],
    format="mixed",
    errors="coerce"
)

In [9]:
dim_patient = dim_patient[
    [
        "PatientKey",
        "PatientID",
        "BirthDate",
        "Gender",
        "Race",
        "Ethnicity",
        "ZipCode",
        "State",
        "Region",
        "RegistrationDate",
        "BirthDateValidFlag"
    ]
]

In [10]:
dim_patient.head()

,PatientKey,PatientID,BirthDate,Gender,Race,Ethnicity,ZipCode,State,Region,RegistrationDate,BirthDateValidFlag
0,0,UNKNOWN,NaT,Unknown,Unknown,Unknown,Unknown,UN,Unknown,NaT,0
1,1,PAT000001,2006-07-24,Male,White,Not Hispanic or Latino,22554,VA,Northern,2023-12-08,1
2,2,PAT000002,1953-04-07,Female,White,Not Hispanic or Latino,23834,VA,Southern,NaT,1
3,3,PAT000003,1994-03-09,Male,White,Not Hispanic or Latino,23606,VA,Eastern,NaT,1
4,4,PAT000004,2023-08-26,Female,Black or African American,Not Hispanic or Latino,22401,VA,Northern,NaT,1


In [11]:
print(
    "Rows:",
    len(dim_patient)
)

print(
    "Duplicate PatientKey:",
    dim_patient[
        "PatientKey"
    ].duplicated().sum()
)

print(
    "Duplicate PatientID:",
    dim_patient[
        "PatientID"
    ].duplicated().sum()
)

print(
    "Missing PatientKey:",
    dim_patient[
        "PatientKey"
    ].isna().sum()
)

print(
    "UNKNOWN PatientKey:",
    dim_patient.loc[
        dim_patient["PatientID"] == "UNKNOWN",
        "PatientKey"
    ].tolist()
)

Rows: 9981
Duplicate PatientKey: 0
Duplicate PatientID: 0
Missing PatientKey: 0
UNKNOWN PatientKey: [0]


In [12]:
dim_patient[
    dim_patient[
        "PatientID"
    ] != "UNKNOWN"
].head(1)

,PatientKey,PatientID,BirthDate,Gender,Race,Ethnicity,ZipCode,State,Region,RegistrationDate,BirthDateValidFlag
1,1,PAT000001,2006-07-24,Male,White,Not Hispanic or Latino,22554,VA,Northern,2023-12-08,1


In [13]:
dim_patient.to_csv(
    WAREHOUSE_DATA_PATH
    / "dim_patient.csv",
    index=False
)

print(
    "dim_patient.csv created successfully."
)

dim_patient.csv created successfully.


### Load staging department data

In [14]:
stg_departments = pd.read_csv(
    STAGING_DATA_PATH / "stg_departments.csv"
)

stg_departments.head()

,DepartmentID,DepartmentName,FacilityName,DepartmentType,City,State
0,UNKNOWN,Unknown / Unmapped,Unknown / Unmapped,Unknown,Unknown,UN
1,DEP001,Emergency Department,RiverCare Central Hospital,Emergency,Richmond,VA
2,DEP002,General Medicine,RiverCare Central Hospital,Inpatient,Richmond,VA
3,DEP003,Cardiology,RiverCare Central Hospital,Specialty,Richmond,VA
4,DEP004,Neurology,RiverCare Central Hospital,Specialty,Richmond,VA


In [15]:
stg_departments = (
    stg_departments
    .assign(
        _UnknownSort=np.where(
            stg_departments["DepartmentID"] == "UNKNOWN",
            0,
            1
        )
    )
    .sort_values(
        ["_UnknownSort", "DepartmentID"]
    )
    .drop(columns="_UnknownSort")
    .reset_index(drop=True)
)

In [16]:
stg_departments.head()

,DepartmentID,DepartmentName,FacilityName,DepartmentType,City,State
0,UNKNOWN,Unknown / Unmapped,Unknown / Unmapped,Unknown,Unknown,UN
1,DEP001,Emergency Department,RiverCare Central Hospital,Emergency,Richmond,VA
2,DEP002,General Medicine,RiverCare Central Hospital,Inpatient,Richmond,VA
3,DEP003,Cardiology,RiverCare Central Hospital,Specialty,Richmond,VA
4,DEP004,Neurology,RiverCare Central Hospital,Specialty,Richmond,VA


### Create DepartmentKey

In [17]:
dim_department = stg_departments.copy()

dim_department.insert(
    0,
    "DepartmentKey",
    range(len(dim_department))
)

In [18]:
dim_department.head(10)

,DepartmentKey,DepartmentID,DepartmentName,FacilityName,DepartmentType,City,State
0,0,UNKNOWN,Unknown / Unmapped,Unknown / Unmapped,Unknown,Unknown,UN
1,1,DEP001,Emergency Department,RiverCare Central Hospital,Emergency,Richmond,VA
2,2,DEP002,General Medicine,RiverCare Central Hospital,Inpatient,Richmond,VA
3,3,DEP003,Cardiology,RiverCare Central Hospital,Specialty,Richmond,VA
4,4,DEP004,Neurology,RiverCare Central Hospital,Specialty,Richmond,VA
5,5,DEP005,Orthopedics,RiverCare Central Hospital,Specialty,Richmond,VA
6,6,DEP006,Oncology,RiverCare Central Hospital,Specialty,Richmond,VA
7,7,DEP007,Emergency Department,RiverCare North Hospital,Emergency,Fredericksburg,VA
8,8,DEP008,General Medicine,RiverCare North Hospital,Inpatient,Fredericksburg,VA
9,9,DEP009,Cardiology,RiverCare North Hospital,Specialty,Fredericksburg,VA


### Keep the final warehouse columns

In [19]:
dim_department = dim_department[
    [
        "DepartmentKey",
        "DepartmentID",
        "DepartmentName",
        "DepartmentType",
        "FacilityName",
        "City",
        "State"
    ]
]

In [20]:
dim_department.head()

,DepartmentKey,DepartmentID,DepartmentName,DepartmentType,FacilityName,City,State
0,0,UNKNOWN,Unknown / Unmapped,Unknown,Unknown / Unmapped,Unknown,UN
1,1,DEP001,Emergency Department,Emergency,RiverCare Central Hospital,Richmond,VA
2,2,DEP002,General Medicine,Inpatient,RiverCare Central Hospital,Richmond,VA
3,3,DEP003,Cardiology,Specialty,RiverCare Central Hospital,Richmond,VA
4,4,DEP004,Neurology,Specialty,RiverCare Central Hospital,Richmond,VA


### Validate it

In [21]:
print(
    "Rows:",
    len(dim_department)
)

print(
    "Duplicate DepartmentKey:",
    dim_department[
        "DepartmentKey"
    ].duplicated().sum()
)

print(
    "Duplicate DepartmentID:",
    dim_department[
        "DepartmentID"
    ].duplicated().sum()
)

print(
    "Missing DepartmentKey:",
    dim_department[
        "DepartmentKey"
    ].isna().sum()
)

print(
    "UNKNOWN DepartmentKey:",
    dim_department.loc[
        dim_department["DepartmentID"] == "UNKNOWN",
        "DepartmentKey"
    ].tolist()
)

Rows: 26
Duplicate DepartmentKey: 0
Duplicate DepartmentID: 0
Missing DepartmentKey: 0
UNKNOWN DepartmentKey: [0]


In [22]:
dim_department.to_csv(
    WAREHOUSE_DATA_PATH
    / "dim_department.csv",
    index=False
)

print(
    "dim_department.csv created successfully."
)

dim_department.csv created successfully.


In [23]:
stg_providers = pd.read_csv(
    STAGING_DATA_PATH / "stg_providers.csv"
)

stg_providers.head()

,ProviderID,DepartmentID,ProviderType,Specialty,HireDate,ActiveFlag
0,UNKNOWN,UNKNOWN,Unknown,Unknown,NaN,0
1,PRV001,DEP003,Nurse Practitioner,Cardiology,2015-07-02,0
2,PRV002,DEP022,Physician,Pediatrics,2023-06-06,0
3,PRV003,DEP024,Physician,Pulmonology,NaN,1
4,PRV004,DEP008,Physician,Internal Medicine,NaN,1


### Convert HireDate

In [24]:
stg_providers["HireDate"] = pd.to_datetime(
    stg_providers["HireDate"],
    format="mixed",
    errors="coerce"
)

### Put UNKNOWN first

In [25]:
stg_providers = (
    stg_providers
    .assign(
        _UnknownSort=np.where(
            stg_providers["ProviderID"] == "UNKNOWN",
            0,
            1
        )
    )
    .sort_values(
        ["_UnknownSort", "ProviderID"]
    )
    .drop(columns="_UnknownSort")
    .reset_index(drop=True)
)

In [26]:
stg_providers.head()

,ProviderID,DepartmentID,ProviderType,Specialty,HireDate,ActiveFlag
0,UNKNOWN,UNKNOWN,Unknown,Unknown,NaT,0
1,PRV001,DEP003,Nurse Practitioner,Cardiology,2015-07-02,0
2,PRV002,DEP022,Physician,Pediatrics,2023-06-06,0
3,PRV003,DEP024,Physician,Pulmonology,NaT,1
4,PRV004,DEP008,Physician,Internal Medicine,NaT,1


### Add ProviderKey

In [27]:
dim_provider = stg_providers.copy()

dim_provider.insert(
    0,
    "ProviderKey",
    range(len(dim_provider))
)

In [28]:
dim_provider.head()

,ProviderKey,ProviderID,DepartmentID,ProviderType,Specialty,HireDate,ActiveFlag
0,0,UNKNOWN,UNKNOWN,Unknown,Unknown,NaT,0
1,1,PRV001,DEP003,Nurse Practitioner,Cardiology,2015-07-02,0
2,2,PRV002,DEP022,Physician,Pediatrics,2023-06-06,0
3,3,PRV003,DEP024,Physician,Pulmonology,NaT,1
4,4,PRV004,DEP008,Physician,Internal Medicine,NaT,1


### Convert DepartmentID into DepartmentKey

In [29]:
department_key_lookup = (
    dim_department[
        [
            "DepartmentID",
            "DepartmentKey"
        ]
    ]
    .copy()
)

In [30]:
dim_provider = dim_provider.merge(
    department_key_lookup,
    on="DepartmentID",
    how="left"
)

In [31]:
dim_provider[
    [
        "ProviderKey",
        "ProviderID",
        "DepartmentID",
        "DepartmentKey"
    ]
].head(10)

,ProviderKey,ProviderID,DepartmentID,DepartmentKey
0,0,UNKNOWN,UNKNOWN,0
1,1,PRV001,DEP003,3
2,2,PRV002,DEP022,22
3,3,PRV003,DEP024,24
4,4,PRV004,DEP008,8
5,5,PRV005,DEP024,24
6,6,PRV006,DEP008,8
7,7,PRV007,DEP024,24
8,8,PRV008,DEP022,22
9,9,PRV009,DEP025,25


### Check for unresolved departments

In [32]:
print(
    "Providers with missing DepartmentKey:",
    dim_provider[
        "DepartmentKey"
    ].isna().sum()
)

Providers with missing DepartmentKey: 0


### Keep the final columns

In [33]:
dim_provider = dim_provider[
    [
        "ProviderKey",
        "ProviderID",
        "DepartmentKey",
        "DepartmentID",
        "ProviderType",
        "Specialty",
        "HireDate",
        "ActiveFlag"
    ]
]

In [34]:
dim_provider.head()

,ProviderKey,ProviderID,DepartmentKey,DepartmentID,ProviderType,Specialty,HireDate,ActiveFlag
0,0,UNKNOWN,0,UNKNOWN,Unknown,Unknown,NaT,0
1,1,PRV001,3,DEP003,Nurse Practitioner,Cardiology,2015-07-02,0
2,2,PRV002,22,DEP022,Physician,Pediatrics,2023-06-06,0
3,3,PRV003,24,DEP024,Physician,Pulmonology,NaT,1
4,4,PRV004,8,DEP008,Physician,Internal Medicine,NaT,1


### Validate DimProvider

In [35]:
print(
    "Rows:",
    len(dim_provider)
)

print(
    "Duplicate ProviderKey:",
    dim_provider[
        "ProviderKey"
    ].duplicated().sum()
)

print(
    "Duplicate ProviderID:",
    dim_provider[
        "ProviderID"
    ].duplicated().sum()
)

print(
    "Missing ProviderKey:",
    dim_provider[
        "ProviderKey"
    ].isna().sum()
)

print(
    "Missing DepartmentKey:",
    dim_provider[
        "DepartmentKey"
    ].isna().sum()
)

print(
    "UNKNOWN ProviderKey:",
    dim_provider.loc[
        dim_provider["ProviderID"] == "UNKNOWN",
        "ProviderKey"
    ].tolist()
)

Rows: 121
Duplicate ProviderKey: 0
Duplicate ProviderID: 0
Missing ProviderKey: 0
Missing DepartmentKey: 0
UNKNOWN ProviderKey: [0]


In [36]:
dim_provider.to_csv(
    WAREHOUSE_DATA_PATH
    / "dim_provider.csv",
    index=False
)

print(
    "dim_provider.csv created successfully."
)

dim_provider.csv created successfully.


### Build DimPayer

### Load staging payer data

In [37]:
stg_payers = pd.read_csv(
    STAGING_DATA_PATH / "stg_payers.csv"
)

stg_payers.head()

,PayerID,PayerName,PayerType
0,UNKNOWN,Unknown / Unmapped,Other
1,PAY01,Medicare,Government
2,PAY02,Medicaid,Government
3,PAY03,Commercial Insurance,Commercial
4,PAY04,Employer Sponsored,Commercial


### Put UNKNOWN first

In [38]:
stg_payers = (
    stg_payers
    .assign(
        _UnknownSort=np.where(
            stg_payers["PayerID"] == "UNKNOWN",
            0,
            1
        )
    )
    .sort_values(
        ["_UnknownSort", "PayerID"]
    )
    .drop(columns="_UnknownSort")
    .reset_index(drop=True)
)

In [39]:
stg_payers.head()

,PayerID,PayerName,PayerType
0,UNKNOWN,Unknown / Unmapped,Other
1,PAY01,Medicare,Government
2,PAY02,Medicaid,Government
3,PAY03,Commercial Insurance,Commercial
4,PAY04,Employer Sponsored,Commercial


### Create PayerKey

In [40]:
dim_payer = stg_payers.copy()

dim_payer.insert(
    0,
    "PayerKey",
    range(len(dim_payer))
)

In [41]:
dim_payer.head()

,PayerKey,PayerID,PayerName,PayerType
0,0,UNKNOWN,Unknown / Unmapped,Other
1,1,PAY01,Medicare,Government
2,2,PAY02,Medicaid,Government
3,3,PAY03,Commercial Insurance,Commercial
4,4,PAY04,Employer Sponsored,Commercial


### Keep final columns

In [42]:
dim_payer = dim_payer[
    [
        "PayerKey",
        "PayerID",
        "PayerName",
        "PayerType"
    ]
]

### Validate

In [43]:
print(
    "Rows:",
    len(dim_payer)
)

print(
    "Duplicate PayerKey:",
    dim_payer[
        "PayerKey"
    ].duplicated().sum()
)

print(
    "Duplicate PayerID:",
    dim_payer[
        "PayerID"
    ].duplicated().sum()
)

print(
    "Missing PayerKey:",
    dim_payer[
        "PayerKey"
    ].isna().sum()
)

print(
    "UNKNOWN PayerKey:",
    dim_payer.loc[
        dim_payer["PayerID"] == "UNKNOWN",
        "PayerKey"
    ].tolist()
)

Rows: 7
Duplicate PayerKey: 0
Duplicate PayerID: 0
Missing PayerKey: 0
UNKNOWN PayerKey: [0]


In [44]:
dim_payer.to_csv(
    WAREHOUSE_DATA_PATH
    / "dim_payer.csv",
    index=False
)

print(
    "dim_payer.csv created successfully."
)

dim_payer.csv created successfully.


## Build DimDiagnosis

### Load staging diagnosis data

In [45]:
stg_diagnoses = pd.read_csv(
    STAGING_DATA_PATH / "stg_diagnoses.csv"
)

stg_diagnoses.head()

,DiagnosisID,DiagnosisCode,DiagnosisName,DiagnosisCategory,ChronicConditionFlag
0,UNKNOWN,UNKNOWN,Unknown / Unmapped Diagnosis,Other,0
1,DX001,E11,Type 2 Diabetes,Diabetes,1
2,DX002,E10,Type 1 Diabetes,Diabetes,1
3,DX003,I10,Essential Hypertension,Hypertension,1
4,DX004,I50,Heart Failure,Heart Disease,1


### Put UNKNOWN first

In [46]:
stg_diagnoses = (
    stg_diagnoses
    .assign(
        _UnknownSort=np.where(
            stg_diagnoses["DiagnosisID"] == "UNKNOWN",
            0,
            1
        )
    )
    .sort_values(
        ["_UnknownSort", "DiagnosisID"]
    )
    .drop(columns="_UnknownSort")
    .reset_index(drop=True)
)

In [47]:
stg_diagnoses.head()

,DiagnosisID,DiagnosisCode,DiagnosisName,DiagnosisCategory,ChronicConditionFlag
0,UNKNOWN,UNKNOWN,Unknown / Unmapped Diagnosis,Other,0
1,DX001,E11,Type 2 Diabetes,Diabetes,1
2,DX002,E10,Type 1 Diabetes,Diabetes,1
3,DX003,I10,Essential Hypertension,Hypertension,1
4,DX004,I50,Heart Failure,Heart Disease,1


### Create DiagnosisKey

In [48]:
dim_diagnosis = stg_diagnoses.copy()

dim_diagnosis.insert(
    0,
    "DiagnosisKey",
    range(len(dim_diagnosis))
)

In [49]:
dim_diagnosis.head()

,DiagnosisKey,DiagnosisID,DiagnosisCode,DiagnosisName,DiagnosisCategory,ChronicConditionFlag
0,0,UNKNOWN,UNKNOWN,Unknown / Unmapped Diagnosis,Other,0
1,1,DX001,E11,Type 2 Diabetes,Diabetes,1
2,2,DX002,E10,Type 1 Diabetes,Diabetes,1
3,3,DX003,I10,Essential Hypertension,Hypertension,1
4,4,DX004,I50,Heart Failure,Heart Disease,1


### Keep final columns

In [50]:
dim_diagnosis = dim_diagnosis[
    [
        "DiagnosisKey",
        "DiagnosisID",
        "DiagnosisCode",
        "DiagnosisName",
        "DiagnosisCategory",
        "ChronicConditionFlag"
    ]
]

In [51]:
print(
    "Rows:",
    len(dim_diagnosis)
)

print(
    "Duplicate DiagnosisKey:",
    dim_diagnosis[
        "DiagnosisKey"
    ].duplicated().sum()
)

print(
    "Duplicate DiagnosisID:",
    dim_diagnosis[
        "DiagnosisID"
    ].duplicated().sum()
)

print(
    "Missing DiagnosisKey:",
    dim_diagnosis[
        "DiagnosisKey"
    ].isna().sum()
)

print(
    "UNKNOWN DiagnosisKey:",
    dim_diagnosis.loc[
        dim_diagnosis["DiagnosisID"] == "UNKNOWN",
        "DiagnosisKey"
    ].tolist()
)

Rows: 51
Duplicate DiagnosisKey: 0
Duplicate DiagnosisID: 0
Missing DiagnosisKey: 0
UNKNOWN DiagnosisKey: [0]


In [52]:
dim_diagnosis.to_csv(
    WAREHOUSE_DATA_PATH
    / "dim_diagnosis.csv",
    index=False
)

print(
    "dim_diagnosis.csv created successfully."
)

dim_diagnosis.csv created successfully.


## Build DimProcedure

### Load staging procedure data

In [53]:
stg_procedures = pd.read_csv(
    STAGING_DATA_PATH / "stg_procedures.csv"
)

stg_procedures.head()

,ProcedureID,ProcedureCode,ProcedureName,ProcedureCategory,StandardCost
0,UNKNOWN,UNKNOWN,Unknown / Unmapped Procedure,Unknown,NaN
1,PROC001,PR001,Complete Blood Count,Laboratory,85.0
2,PROC002,PR002,Basic Metabolic Panel,Laboratory,95.0
3,PROC003,PR003,HbA1c Test,Laboratory,70.0
4,PROC004,PR004,Lipid Panel,Laboratory,90.0


In [54]:
stg_procedures = (
    stg_procedures
    .assign(
        _UnknownSort=np.where(
            stg_procedures["ProcedureID"] == "UNKNOWN",
            0,
            1
        )
    )
    .sort_values(
        ["_UnknownSort", "ProcedureID"]
    )
    .drop(columns="_UnknownSort")
    .reset_index(drop=True)
)

In [55]:
stg_procedures.head()

,ProcedureID,ProcedureCode,ProcedureName,ProcedureCategory,StandardCost
0,UNKNOWN,UNKNOWN,Unknown / Unmapped Procedure,Unknown,NaN
1,PROC001,PR001,Complete Blood Count,Laboratory,85.0
2,PROC002,PR002,Basic Metabolic Panel,Laboratory,95.0
3,PROC003,PR003,HbA1c Test,Laboratory,70.0
4,PROC004,PR004,Lipid Panel,Laboratory,90.0


In [56]:
dim_procedure = stg_procedures.copy()

dim_procedure.insert(
    0,
    "ProcedureKey",
    range(len(dim_procedure))
)

In [57]:
dim_procedure.head()

,ProcedureKey,ProcedureID,ProcedureCode,ProcedureName,ProcedureCategory,StandardCost
0,0,UNKNOWN,UNKNOWN,Unknown / Unmapped Procedure,Unknown,NaN
1,1,PROC001,PR001,Complete Blood Count,Laboratory,85.0
2,2,PROC002,PR002,Basic Metabolic Panel,Laboratory,95.0
3,3,PROC003,PR003,HbA1c Test,Laboratory,70.0
4,4,PROC004,PR004,Lipid Panel,Laboratory,90.0


In [58]:
dim_procedure = dim_procedure[
    [
        "ProcedureKey",
        "ProcedureID",
        "ProcedureCode",
        "ProcedureName",
        "ProcedureCategory",
        "StandardCost"
    ]
]

In [59]:
print(
    "Rows:",
    len(dim_procedure)
)

print(
    "Duplicate ProcedureKey:",
    dim_procedure[
        "ProcedureKey"
    ].duplicated().sum()
)

print(
    "Duplicate ProcedureID:",
    dim_procedure[
        "ProcedureID"
    ].duplicated().sum()
)

print(
    "Missing ProcedureKey:",
    dim_procedure[
        "ProcedureKey"
    ].isna().sum()
)

print(
    "UNKNOWN ProcedureKey:",
    dim_procedure.loc[
        dim_procedure["ProcedureID"] == "UNKNOWN",
        "ProcedureKey"
    ].tolist()
)

Rows: 41
Duplicate ProcedureKey: 0
Duplicate ProcedureID: 0
Missing ProcedureKey: 0
UNKNOWN ProcedureKey: [0]


In [60]:
dim_procedure.to_csv(
    WAREHOUSE_DATA_PATH
    / "dim_procedure.csv",
    index=False
)

print(
    "dim_procedure.csv created successfully."
)

dim_procedure.csv created successfully.


### Build DimDate

In [61]:
stg_encounters = pd.read_csv(
    STAGING_DATA_PATH / "stg_encounters.csv"
)

stg_admissions = pd.read_csv(
    STAGING_DATA_PATH / "stg_admissions.csv"
)

stg_appointments = pd.read_csv(
    STAGING_DATA_PATH / "stg_appointments.csv"
)

stg_encounter_procedures = pd.read_csv(
    STAGING_DATA_PATH / "stg_encounter_procedures.csv"
)

stg_lab_results = pd.read_csv(
    STAGING_DATA_PATH / "stg_lab_results.csv"
)

print("Date-related staging tables loaded.")

Date-related staging tables loaded.


In [62]:
date_columns = [
    (stg_encounters, "EncounterDate"),

    (stg_admissions, "AdmissionDateTime"),
    (stg_admissions, "DischargeDateTime"),
    (stg_admissions, "FollowUpDate"),

    (stg_appointments, "ScheduledDate"),
    (stg_appointments, "AppointmentDateTime"),

    (stg_encounter_procedures, "ProcedureDate"),

    (stg_lab_results, "ResultDateTime")
]

In [63]:
all_min_dates = []
all_max_dates = []

for df, column in date_columns:

    parsed_dates = pd.to_datetime(
        df[column],
        format="mixed",
        errors="coerce"
    )

    if parsed_dates.notna().any():

        all_min_dates.append(
            parsed_dates.min()
        )

        all_max_dates.append(
            parsed_dates.max()
        )

min_date = min(all_min_dates).normalize()
max_date = max(all_max_dates).normalize()

print("Earliest date:", min_date)
print("Latest date:", max_date)

Earliest date: 2023-01-02 00:00:00
Latest date: 2026-02-05 00:00:00


In [64]:
date_range = pd.date_range(
    start=min_date,
    end=max_date,
    freq="D"
)

dim_date = pd.DataFrame({
    "FullDate": date_range
})

In [65]:
dim_date.head()

,FullDate
0,2023-01-02
1,2023-01-03
2,2023-01-04
3,2023-01-05
4,2023-01-06


In [66]:
dim_date["DateKey"] = (
    dim_date["FullDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [67]:
dim_date[
    [
        "DateKey",
        "FullDate"
    ]
].head()

,DateKey,FullDate
0,20230102,2023-01-02
1,20230103,2023-01-03
2,20230104,2023-01-04
3,20230105,2023-01-05
4,20230106,2023-01-06


In [68]:
dim_date["Day"] = (
    dim_date["FullDate"].dt.day
)

dim_date["DayName"] = (
    dim_date["FullDate"].dt.day_name()
)

dim_date["DayOfWeek"] = (
    dim_date["FullDate"].dt.dayofweek + 1
)

dim_date["WeekOfYear"] = (
    dim_date["FullDate"]
    .dt.isocalendar()
    .week
    .astype(int)
)

dim_date["Month"] = (
    dim_date["FullDate"].dt.month
)

dim_date["MonthName"] = (
    dim_date["FullDate"].dt.month_name()
)

dim_date["Quarter"] = (
    "Q"
    + dim_date["FullDate"]
    .dt.quarter
    .astype(str)
)

dim_date["Year"] = (
    dim_date["FullDate"].dt.year
)

dim_date["YearMonth"] = (
    dim_date["FullDate"]
    .dt.strftime("%Y-%m")
)

dim_date["IsWeekend"] = (
    dim_date["FullDate"]
    .dt.dayofweek
    .isin([5, 6])
    .astype(int)
)

In [69]:
dim_date.head()

,FullDate,DateKey,Day,DayName,DayOfWeek,WeekOfYear,Month,MonthName,Quarter,Year,YearMonth,IsWeekend
0,2023-01-02,20230102,2,Monday,1,1,1,January,Q1,2023,2023-01,0
1,2023-01-03,20230103,3,Tuesday,2,1,1,January,Q1,2023,2023-01,0
2,2023-01-04,20230104,4,Wednesday,3,1,1,January,Q1,2023,2023-01,0
3,2023-01-05,20230105,5,Thursday,4,1,1,January,Q1,2023,2023-01,0
4,2023-01-06,20230106,6,Friday,5,1,1,January,Q1,2023,2023-01,0


In [70]:
unknown_date = pd.DataFrame([
    {
        "DateKey": 0,
        "FullDate": pd.NaT,
        "Day": 0,
        "DayName": "Unknown",
        "DayOfWeek": 0,
        "WeekOfYear": 0,
        "Month": 0,
        "MonthName": "Unknown",
        "Quarter": "Unknown",
        "Year": 0,
        "YearMonth": "Unknown",
        "IsWeekend": 0
    }
])

In [71]:
dim_date = pd.concat(
    [
        unknown_date,
        dim_date
    ],
    ignore_index=True
)

In [72]:
dim_date.head()

,DateKey,FullDate,Day,DayName,DayOfWeek,WeekOfYear,Month,MonthName,Quarter,Year,YearMonth,IsWeekend
0,0,NaT,0,Unknown,0,0,0,Unknown,Unknown,0,Unknown,0
1,20230102,2023-01-02,2,Monday,1,1,1,January,Q1,2023,2023-01,0
2,20230103,2023-01-03,3,Tuesday,2,1,1,January,Q1,2023,2023-01,0
3,20230104,2023-01-04,4,Wednesday,3,1,1,January,Q1,2023,2023-01,0
4,20230105,2023-01-05,5,Thursday,4,1,1,January,Q1,2023,2023-01,0


In [73]:
dim_date = dim_date[
    [
        "DateKey",
        "FullDate",
        "Day",
        "DayName",
        "DayOfWeek",
        "WeekOfYear",
        "Month",
        "MonthName",
        "Quarter",
        "Year",
        "YearMonth",
        "IsWeekend"
    ]
]

In [74]:
print(
    "Rows:",
    len(dim_date)
)

print(
    "Duplicate DateKey:",
    dim_date[
        "DateKey"
    ].duplicated().sum()
)

print(
    "Missing DateKey:",
    dim_date[
        "DateKey"
    ].isna().sum()
)

print(
    "UNKNOWN DateKey:",
    dim_date.loc[
        dim_date["DateKey"] == 0,
        "DateKey"
    ].tolist()
)

print(
    "First real date:",
    dim_date.loc[
        dim_date["DateKey"] != 0,
        "FullDate"
    ].min()
)

print(
    "Last real date:",
    dim_date.loc[
        dim_date["DateKey"] != 0,
        "FullDate"
    ].max()
)

Rows: 1132
Duplicate DateKey: 0
Missing DateKey: 0
UNKNOWN DateKey: [0]
First real date: 2023-01-02 00:00:00
Last real date: 2026-02-05 00:00:00


In [75]:
dim_date.to_csv(
    WAREHOUSE_DATA_PATH
    / "dim_date.csv",
    index=False
)

print(
    "dim_date.csv created successfully."
)

dim_date.csv created successfully.


### Build FactEncounter

In [76]:
stg_encounters = pd.read_csv(
    STAGING_DATA_PATH / "stg_encounters.csv"
)

print(
    "Staging encounters:",
    len(stg_encounters)
)

stg_encounters.head()

Staging encounters: 90000


,EncounterID,PatientID,ProviderID,DepartmentID,PayerID,EncounterDate,EncounterType,ArrivalDateTime,TriageDateTime,ProviderStartDateTime,...,EncounterQualityFlag,DepartmentIDRecoveredFlag,DepartmentIDResolvedFlag,EncounterEndRecoveredFlag,EncounterEndRecoveryMethod,SourcePatientID,SourceProviderID,SourceDepartmentID,SourceEncounterCost,SourcePayerID
0,ENC000001,PAT003057,PRV051,DEP007,PAY03,2024-06-19,Emergency,2024-06-19 02:35:00,2024-06-19 02:52:00,2024-06-19 03:06:00,...,1,0,1,0,NaN,PAT003057,PRV051,DEP007,1067.62,PAY03
1,ENC000002,PAT005801,UNKNOWN,DEP020,PAY01,2025-11-25,Outpatient,2025-11-25 13:46:00,NaN,2025-11-25 14:00:00,...,1,0,1,0,NaN,PAT005801,NaN,DEP020,385.01,PAY01
2,ENC000003,PAT009821,PRV118,DEP001,PAY06,2025-10-01,Emergency,2025-10-01 18:12:00,2025-10-01 18:36:00,2025-10-01 20:03:00,...,1,0,1,0,NaN,PAT009821,PRV118,DEP001,2056.60,PAY06
3,ENC000004,PAT002460,PRV032,DEP014,PAY05,2023-05-11,Specialist,2023-05-11 14:45:00,NaN,2023-05-11 15:00:00,...,1,0,1,0,NaN,PAT002460,PRV032,DEP014,567.31,PAY05
4,ENC000005,PAT003941,PRV008,DEP022,PAY03,2025-12-16,Outpatient,2025-12-16 15:46:00,NaN,2025-12-16 16:28:00,...,1,0,1,0,NaN,PAT003941,PRV008,DEP022,258.00,PAY03


In [77]:
stg_encounters.columns.tolist()

['EncounterID',
 'PatientID',
 'ProviderID',
 'DepartmentID',
 'PayerID',
 'EncounterDate',
 'EncounterType',
 'ArrivalDateTime',
 'TriageDateTime',
 'ProviderStartDateTime',
 'EncounterEndDateTime',
 'EncounterStatus',
 'EncounterCost',
 'PatientIDValidFlag',
 'DepartmentIDValidFlag',
 'ProviderIDValidFlag',
 'EncounterCostValidFlag',
 'WaitTimeValidFlag',
 'WaitMinutes',
 'EncounterDurationValidFlag',
 'EncounterDurationHours',
 'EncounterQualityFlag',
 'DepartmentIDRecoveredFlag',
 'DepartmentIDResolvedFlag',
 'EncounterEndRecoveredFlag',
 'EncounterEndRecoveryMethod',
 'SourcePatientID',
 'SourceProviderID',
 'SourceDepartmentID',
 'SourceEncounterCost',
 'SourcePayerID']

### Create the fact table

In [78]:
fact_encounter = stg_encounters.copy()

### PatientID → PatientKe

In [79]:
patient_key_map = (
    dim_patient
    .set_index("PatientID")[
        "PatientKey"
    ]
)

In [80]:
fact_encounter["PatientKey"] = (
    fact_encounter["PatientID"]
    .map(patient_key_map)
    .fillna(0)
    .astype(int)
)

### ProviderID → ProviderKey

In [81]:
provider_key_map = (
    dim_provider
    .set_index("ProviderID")[
        "ProviderKey"
    ]
)

fact_encounter["ProviderKey"] = (
    fact_encounter["ProviderID"]
    .map(provider_key_map)
    .fillna(0)
    .astype(int)
)

### DepartmentID → DepartmentKey

In [82]:
department_key_map = (
    dim_department
    .set_index("DepartmentID")[
        "DepartmentKey"
    ]
)

fact_encounter["DepartmentKey"] = (
    fact_encounter["DepartmentID"]
    .map(department_key_map)
    .fillna(0)
    .astype(int)
)

### PayerID → PayerKey

In [83]:
payer_key_map = (
    dim_payer
    .set_index("PayerID")[
        "PayerKey"
    ]
)

fact_encounter["PayerKey"] = (
    fact_encounter["PayerID"]
    .map(payer_key_map)
    .fillna(0)
    .astype(int)
)

### EncounterDate → DateKey

In [84]:
fact_encounter["EncounterDate"] = pd.to_datetime(
    fact_encounter["EncounterDate"],
    format="mixed",
    errors="coerce"
)

In [85]:
fact_encounter["EncounterDateKey"] = (
    fact_encounter[
        "EncounterDate"
    ]
    .dt.strftime("%Y%m%d")
)

In [86]:
fact_encounter["EncounterDateKey"] = (
    pd.to_numeric(
        fact_encounter["EncounterDateKey"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

### Create EncounterKey

In [87]:
fact_encounter = (
    fact_encounter
    .sort_values("EncounterID")
    .reset_index(drop=True)
)

fact_encounter.insert(
    0,
    "EncounterKey",
    range(
        1,
        len(fact_encounter) + 1
    )
)

In [88]:
fact_encounter[
    [
        "EncounterKey",
        "EncounterID",
        "PatientKey",
        "ProviderKey",
        "DepartmentKey",
        "PayerKey",
        "EncounterDateKey"
    ]
].head(10)

,EncounterKey,EncounterID,PatientKey,ProviderKey,DepartmentKey,PayerKey,EncounterDateKey
0,1,ENC000001,3051,51,7,3,20240619
1,2,ENC000002,5787,0,20,1,20251125
2,3,ENC000003,9802,118,1,6,20251001
3,4,ENC000004,2454,32,14,5,20230511
4,5,ENC000005,3934,8,22,3,20251216
5,6,ENC000006,5481,14,9,5,20241011
6,7,ENC000007,7592,45,16,2,20240723
7,8,ENC000008,3649,89,4,1,20230709
8,9,ENC000009,8933,118,1,4,20230711
9,10,ENC000010,6226,105,14,5,20241018


### Convert timestamps

In [89]:
datetime_columns = [
    "ArrivalDateTime",
    "TriageDateTime",
    "ProviderStartDateTime",
    "EncounterEndDateTime"
]

for column in datetime_columns:

    fact_encounter[column] = pd.to_datetime(
        fact_encounter[column],
        format="mixed",
        errors="coerce"
    )

### Build the final FactEncounter

In [90]:
fact_encounter_columns = [
    "EncounterKey",
    "EncounterID",

    "PatientKey",
    "ProviderKey",
    "DepartmentKey",
    "PayerKey",
    "EncounterDateKey",

    "EncounterType",
    "EncounterStatus",

    "ArrivalDateTime",
    "TriageDateTime",
    "ProviderStartDateTime",
    "EncounterEndDateTime",

    "EncounterCost",
    "WaitMinutes",
    "EncounterDurationHours"
]

In [91]:
quality_columns = [
    "PatientIDValidFlag",
    "ProviderIDValidFlag",
    "DepartmentIDValidFlag",
    "DepartmentIDRecoveredFlag",
    "DepartmentIDResolvedFlag",
    "EncounterCostValidFlag",
    "WaitTimeValidFlag",
    "EncounterDurationValidFlag",
    "EncounterEndRecoveredFlag",
    "EncounterQualityFlag"
]

In [92]:
quality_columns = [
    column
    for column in quality_columns
    if column in fact_encounter.columns
]

fact_encounter = fact_encounter[
    fact_encounter_columns
    + quality_columns
].copy()

In [93]:
fact_encounter.head()

,EncounterKey,EncounterID,PatientKey,ProviderKey,DepartmentKey,PayerKey,EncounterDateKey,EncounterType,EncounterStatus,ArrivalDateTime,...,PatientIDValidFlag,ProviderIDValidFlag,DepartmentIDValidFlag,DepartmentIDRecoveredFlag,DepartmentIDResolvedFlag,EncounterCostValidFlag,WaitTimeValidFlag,EncounterDurationValidFlag,EncounterEndRecoveredFlag,EncounterQualityFlag
0,1,ENC000001,3051,51,7,3,20240619,Emergency,Completed,2024-06-19 02:35:00,...,1,1.0,1,0,1,1,1,1,0,1
1,2,ENC000002,5787,0,20,1,20251125,Outpatient,Completed,2025-11-25 13:46:00,...,1,NaN,1,0,1,1,1,1,0,1
2,3,ENC000003,9802,118,1,6,20251001,Emergency,Completed,2025-10-01 18:12:00,...,1,1.0,1,0,1,1,1,1,0,1
3,4,ENC000004,2454,32,14,5,20230511,Specialist,Completed,2023-05-11 14:45:00,...,1,1.0,1,0,1,1,1,1,0,1
4,5,ENC000005,3934,8,22,3,20251216,Outpatient,Completed,2025-12-16 15:46:00,...,1,1.0,1,0,1,1,1,1,0,1


In [94]:
print(
    "FactEncounter rows:",
    len(fact_encounter)
)

print(
    "Unique EncounterIDs:",
    fact_encounter[
        "EncounterID"
    ].nunique()
)

print(
    "Duplicate EncounterID:",
    fact_encounter[
        "EncounterID"
    ].duplicated().sum()
)

print(
    "Duplicate EncounterKey:",
    fact_encounter[
        "EncounterKey"
    ].duplicated().sum()
)

FactEncounter rows: 90000
Unique EncounterIDs: 90000
Duplicate EncounterID: 0
Duplicate EncounterKey: 0


### Validate the foreign keys

In [95]:
print(
    "Missing PatientKey:",
    fact_encounter[
        "PatientKey"
    ].isna().sum()
)

print(
    "Missing ProviderKey:",
    fact_encounter[
        "ProviderKey"
    ].isna().sum()
)

print(
    "Missing DepartmentKey:",
    fact_encounter[
        "DepartmentKey"
    ].isna().sum()
)

print(
    "Missing PayerKey:",
    fact_encounter[
        "PayerKey"
    ].isna().sum()
)

print(
    "Missing EncounterDateKey:",
    fact_encounter[
        "EncounterDateKey"
    ].isna().sum()
)

Missing PatientKey: 0
Missing ProviderKey: 0
Missing DepartmentKey: 0
Missing PayerKey: 0
Missing EncounterDateKey: 0


### Validate that DateKeys exist in DimDate

In [96]:
print(
    "Invalid EncounterDateKey:",
    (
        ~fact_encounter[
            "EncounterDateKey"
        ].isin(
            dim_date[
                "DateKey"
            ]
        )
    ).sum()
)

Invalid EncounterDateKey: 0


### Check how many UNKNOWN relationships exist

In [97]:
print(
    "UNKNOWN Patient:",
    (
        fact_encounter[
            "PatientKey"
        ] == 0
    ).sum()
)

print(
    "UNKNOWN Provider:",
    (
        fact_encounter[
            "ProviderKey"
        ] == 0
    ).sum()
)

print(
    "UNKNOWN Department:",
    (
        fact_encounter[
            "DepartmentKey"
        ] == 0
    ).sum()
)

print(
    "UNKNOWN Payer:",
    (
        fact_encounter[
            "PayerKey"
        ] == 0
    ).sum()
)

print(
    "UNKNOWN Date:",
    (
        fact_encounter[
            "EncounterDateKey"
        ] == 0
    ).sum()
)

UNKNOWN Patient: 176
UNKNOWN Provider: 1401
UNKNOWN Department: 216
UNKNOWN Payer: 0
UNKNOWN Date: 0


In [98]:
fact_encounter.to_csv(
    WAREHOUSE_DATA_PATH
    / "fact_encounter.csv",
    index=False
)

print(
    "fact_encounter.csv created successfully."
)

fact_encounter.csv created successfully.


### Build FactAdmission

### Load the staging admissions

In [99]:
stg_admissions = pd.read_csv(
    STAGING_DATA_PATH / "stg_admissions.csv"
)

print(
    "Staging admissions:",
    len(stg_admissions)
)

stg_admissions.head()

Staging admissions: 17740


,AdmissionID,EncounterID,PatientID,DepartmentID,AdmissionDateTime,DischargeDateTime,AdmissionType,DischargeDisposition,FollowUpRequiredFlag,FollowUpCompletedFlag,FollowUpDate,LOSValidFlag,LengthOfStayDays,FollowUpDateValidFlag,DaysToFollowUp,AdmissionQualityFlag
0,ADM000001,ENC000016,PAT005222,DEP013,2024-08-03 08:14:00,2024-08-07 06:03:56.857084,Emergency,Skilled Nursing Facility,1,1,2024-09-06,1,3.909686,1,29.747259,1
1,ADM000002,ENC000024,PAT001049,DEP002,2024-10-27 21:25:00,2024-11-03 17:05:12.067328,Emergency,Home,1,0,NaN,1,6.819584,1,NaN,1
2,ADM000003,ENC000026,PAT004167,DEP002,2025-02-26 10:32:00,2025-03-05 05:48:53.130791,Elective,Home,1,0,NaN,1,6.803393,1,NaN,1
3,ADM000004,ENC000039,PAT006068,DEP002,2025-12-26 09:16:00,2025-12-28 22:47:58.253861,Elective,Home,1,1,2026-01-04,1,2.563869,1,6.050020,1
4,ADM000005,ENC000040,PAT001095,DEP013,2025-09-27 20:01:00,2025-10-04 05:23:25.600755,Emergency,Home,1,0,NaN,1,6.390574,1,NaN,1


In [100]:
stg_admissions.columns.tolist()

['AdmissionID',
 'EncounterID',
 'PatientID',
 'DepartmentID',
 'AdmissionDateTime',
 'DischargeDateTime',
 'AdmissionType',
 'DischargeDisposition',
 'FollowUpRequiredFlag',
 'FollowUpCompletedFlag',
 'FollowUpDate',
 'LOSValidFlag',
 'LengthOfStayDays',
 'FollowUpDateValidFlag',
 'DaysToFollowUp',
 'AdmissionQualityFlag']

### Start the fact table

In [101]:
fact_admission = stg_admissions.copy()

### PatientID → PatientKey

In [102]:
patient_key_map = (
    dim_patient
    .set_index("PatientID")["PatientKey"]
)

fact_admission["PatientKey"] = (
    fact_admission["PatientID"]
    .map(patient_key_map)
    .fillna(0)
    .astype(int)
)

### DepartmentID → DepartmentKey

In [103]:
department_key_map = (
    dim_department
    .set_index("DepartmentID")["DepartmentKey"]
)

fact_admission["DepartmentKey"] = (
    fact_admission["DepartmentID"]
    .map(department_key_map)
    .fillna(0)
    .astype(int)
)

### EncounterID → EncounterKey

In [104]:
encounter_key_map = (
    fact_encounter
    .set_index("EncounterID")["EncounterKey"]
)

In [105]:
fact_admission["EncounterKey"] = (
    fact_admission["EncounterID"]
    .map(encounter_key_map)
)

In [106]:
print(
    "Admissions without matching EncounterKey:",
    fact_admission[
        "EncounterKey"
    ].isna().sum()
)

Admissions without matching EncounterKey: 0


### Convert the admission dates

In [107]:
date_columns = [
    "AdmissionDateTime",
    "DischargeDateTime",
    "FollowUpDate"
]

for column in date_columns:

    fact_admission[column] = pd.to_datetime(
        fact_admission[column],
        format="mixed",
        errors="coerce"
    )

### Create AdmissionDateKey

In [108]:
fact_admission["AdmissionDateKey"] = (
    pd.to_numeric(
        fact_admission[
            "AdmissionDateTime"
        ].dt.strftime("%Y%m%d"),
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

### Create DischargeDateKey

In [109]:
fact_admission["DischargeDateKey"] = (
    pd.to_numeric(
        fact_admission[
            "DischargeDateTime"
        ].dt.strftime("%Y%m%d"),
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

### Create FollowUpDateKey

In [110]:
fact_admission["FollowUpDateKey"] = (
    pd.to_numeric(
        fact_admission[
            "FollowUpDate"
        ].dt.strftime("%Y%m%d"),
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

### Create AdmissionKey

In [111]:
fact_admission = (
    fact_admission
    .sort_values("AdmissionID")
    .reset_index(drop=True)
)

fact_admission.insert(
    0,
    "AdmissionKey",
    range(
        1,
        len(fact_admission) + 1
    )
)

In [112]:
fact_admission[
    [
        "AdmissionKey",
        "AdmissionID",
        "EncounterKey",
        "PatientKey",
        "DepartmentKey",
        "AdmissionDateKey",
        "DischargeDateKey",
        "FollowUpDateKey"
    ]
].head(10)

,AdmissionKey,AdmissionID,EncounterKey,PatientKey,DepartmentKey,AdmissionDateKey,DischargeDateKey,FollowUpDateKey
0,1,ADM000001,16,5212,13,20240803,20240807,20240906
1,2,ADM000002,24,1044,2,20241027,20241103,0
2,3,ADM000003,26,4160,2,20250226,20250305,0
3,4,ADM000004,39,6054,2,20251226,20251228,20260104
4,5,ADM000005,40,1090,13,20250927,20251004,0
5,6,ADM000006,41,6820,8,20250823,20250825,0
6,7,ADM000007,51,9848,2,20251213,20251216,20251223
7,8,ADM000008,56,358,8,20251231,20260103,20260124
8,9,ADM000009,60,1384,8,20240503,20240509,0
9,10,ADM000010,63,6058,13,20231227,20240101,20240115


### Build final FactAdmission

In [113]:
fact_admission_columns = [
    "AdmissionKey",
    "AdmissionID",
    "EncounterKey",

    "PatientKey",
    "DepartmentKey",

    "AdmissionDateKey",
    "DischargeDateKey",
    "FollowUpDateKey",

    "AdmissionDateTime",
    "DischargeDateTime",

    "AdmissionType",
    "DischargeDisposition",

    "LengthOfStayDays",

    "FollowUpRequiredFlag",
    "FollowUpCompletedFlag",
    "DaysToFollowUp"
]

In [114]:
quality_columns = [
    "LOSValidFlag",
    "FollowUpDateValidFlag",
    "AdmissionQualityFlag"
]

quality_columns = [
    column
    for column in quality_columns
    if column in fact_admission.columns
]

In [115]:
fact_admission = fact_admission[
    fact_admission_columns
    + quality_columns
].copy()

In [116]:
fact_admission.head()

,AdmissionKey,AdmissionID,EncounterKey,PatientKey,DepartmentKey,AdmissionDateKey,DischargeDateKey,FollowUpDateKey,AdmissionDateTime,DischargeDateTime,AdmissionType,DischargeDisposition,LengthOfStayDays,FollowUpRequiredFlag,FollowUpCompletedFlag,DaysToFollowUp,LOSValidFlag,FollowUpDateValidFlag,AdmissionQualityFlag
0,1,ADM000001,16,5212,13,20240803,20240807,20240906,2024-08-03 08:14:00,2024-08-07 06:03:56.857084,Emergency,Skilled Nursing Facility,3.909686,1,1,29.747259,1,1,1
1,2,ADM000002,24,1044,2,20241027,20241103,0,2024-10-27 21:25:00,2024-11-03 17:05:12.067328,Emergency,Home,6.819584,1,0,NaN,1,1,1
2,3,ADM000003,26,4160,2,20250226,20250305,0,2025-02-26 10:32:00,2025-03-05 05:48:53.130791,Elective,Home,6.803393,1,0,NaN,1,1,1
3,4,ADM000004,39,6054,2,20251226,20251228,20260104,2025-12-26 09:16:00,2025-12-28 22:47:58.253861,Elective,Home,2.563869,1,1,6.050020,1,1,1
4,5,ADM000005,40,1090,13,20250927,20251004,0,2025-09-27 20:01:00,2025-10-04 05:23:25.600755,Emergency,Home,6.390574,1,0,NaN,1,1,1


### Validate the grain

In [117]:
print(
    "FactAdmission rows:",
    len(fact_admission)
)

print(
    "Unique AdmissionIDs:",
    fact_admission[
        "AdmissionID"
    ].nunique()
)

print(
    "Duplicate AdmissionID:",
    fact_admission[
        "AdmissionID"
    ].duplicated().sum()
)

print(
    "Duplicate AdmissionKey:",
    fact_admission[
        "AdmissionKey"
    ].duplicated().sum()
)

FactAdmission rows: 17740
Unique AdmissionIDs: 17740
Duplicate AdmissionID: 0
Duplicate AdmissionKey: 0


### Check all dimension keys

In [118]:
print(
    "Missing PatientKey:",
    fact_admission[
        "PatientKey"
    ].isna().sum()
)

print(
    "Missing DepartmentKey:",
    fact_admission[
        "DepartmentKey"
    ].isna().sum()
)

print(
    "Missing EncounterKey:",
    fact_admission[
        "EncounterKey"
    ].isna().sum()
)

Missing PatientKey: 0
Missing DepartmentKey: 0
Missing EncounterKey: 0


### Validate date relationships

In [119]:
for column in [
    "AdmissionDateKey",
    "DischargeDateKey",
    "FollowUpDateKey"
]:

    invalid_count = (
        ~fact_admission[column]
        .isin(dim_date["DateKey"])
    ).sum()

    print(
        column,
        "invalid keys:",
        invalid_count
    )

AdmissionDateKey invalid keys: 0
DischargeDateKey invalid keys: 0
FollowUpDateKey invalid keys: 0


### Check LOS

In [120]:
print(
    "Negative LengthOfStayDays:",
    (
        fact_admission[
            "LengthOfStayDays"
        ] < 0
    ).sum()
)

Negative LengthOfStayDays: 0


In [121]:
fact_admission.to_csv(
    WAREHOUSE_DATA_PATH
    / "fact_admission.csv",
    index=False
)

print(
    "fact_admission.csv created successfully."
)

fact_admission.csv created successfully.


### Build FactAppointment

In [122]:
stg_appointments = pd.read_csv(
    STAGING_DATA_PATH / "stg_appointments.csv"
)

print(
    "Staging appointments:",
    len(stg_appointments)
)

stg_appointments.head()

Staging appointments: 50000


,AppointmentID,PatientID,ProviderID,DepartmentID,ScheduledDate,AppointmentDateTime,AppointmentStatus,AppointmentType,CancellationReason,PatientIDValidFlag,...,ProviderIDValidFlag,AppointmentStatusValidFlag,BookingLeadTimeValidFlag,BookingLeadDays,CancellationReasonValidFlag,SourcePatientID,SourceProviderID,SourceDepartmentID,SourceAppointmentStatus,SourceCancellationReason
0,APT000001,PAT001942,PRV023,DEP025,2025-11-11,2025-11-23 16:00:00,Completed,Post-Discharge Follow-Up,NaN,1,...,1,1,1,12.0,1,PAT001942,PRV023,DEP025,Completed,NaN
1,APT000002,PAT009810,PRV074,DEP004,2025-08-15,2025-08-29 16:45:00,Completed,Routine,NaN,1,...,1,1,1,14.0,1,PAT009810,PRV074,DEP004,Completed,NaN
2,APT000003,PAT008784,PRV057,DEP021,2025-04-15,2025-06-10 14:00:00,Completed,Specialist,NaN,1,...,1,1,1,56.0,1,PAT008784,PRV057,DEP021,Completed,NaN
3,APT000004,PAT004494,PRV106,DEP019,2024-09-03,2024-11-01 16:15:00,Cancelled,Specialist,Scheduling Conflict,1,...,1,1,1,59.0,1,PAT004494,PRV106,DEP019,Cancelled,Scheduling Conflict
4,APT000005,PAT009138,PRV116,DEP009,2025-06-01,2025-06-04 13:45:00,Completed,Follow-Up,NaN,1,...,1,1,1,3.0,1,PAT009138,PRV116,DEP009,Completed,NaN


In [123]:
stg_appointments.columns.tolist()

['AppointmentID',
 'PatientID',
 'ProviderID',
 'DepartmentID',
 'ScheduledDate',
 'AppointmentDateTime',
 'AppointmentStatus',
 'AppointmentType',
 'CancellationReason',
 'PatientIDValidFlag',
 'DepartmentIDValidFlag',
 'ProviderIDValidFlag',
 'AppointmentStatusValidFlag',
 'BookingLeadTimeValidFlag',
 'BookingLeadDays',
 'CancellationReasonValidFlag',
 'SourcePatientID',
 'SourceProviderID',
 'SourceDepartmentID',
 'SourceAppointmentStatus',
 'SourceCancellationReason']

### Start the fact table

In [124]:
fact_appointment = stg_appointments.copy()

### PatientID → PatientKey

In [125]:
patient_key_map = (
    dim_patient
    .set_index("PatientID")["PatientKey"]
)

fact_appointment["PatientKey"] = (
    fact_appointment["PatientID"]
    .map(patient_key_map)
    .fillna(0)
    .astype(int)
)

### ProviderID → ProviderKey

In [126]:
provider_key_map = (
    dim_provider
    .set_index("ProviderID")["ProviderKey"]
)

fact_appointment["ProviderKey"] = (
    fact_appointment["ProviderID"]
    .map(provider_key_map)
    .fillna(0)
    .astype(int)
)

### DepartmentID → DepartmentKey

In [127]:
department_key_map = (
    dim_department
    .set_index("DepartmentID")["DepartmentKey"]
)

fact_appointment["DepartmentKey"] = (
    fact_appointment["DepartmentID"]
    .map(department_key_map)
    .fillna(0)
    .astype(int)
)

### Convert the date fields

In [128]:
fact_appointment["ScheduledDate"] = pd.to_datetime(
    fact_appointment["ScheduledDate"],
    format="mixed",
    errors="coerce"
)

fact_appointment["AppointmentDateTime"] = pd.to_datetime(
    fact_appointment["AppointmentDateTime"],
    format="mixed",
    errors="coerce"
)

### Create ScheduledDateKey

In [129]:
fact_appointment["ScheduledDateKey"] = (
    pd.to_numeric(
        fact_appointment[
            "ScheduledDate"
        ].dt.strftime("%Y%m%d"),
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

### Create AppointmentDateKey

In [130]:
fact_appointment["AppointmentDateKey"] = (
    pd.to_numeric(
        fact_appointment[
            "AppointmentDateTime"
        ].dt.strftime("%Y%m%d"),
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

### Create AppointmentKey

In [131]:
fact_appointment = (
    fact_appointment
    .sort_values("AppointmentID")
    .reset_index(drop=True)
)

fact_appointment.insert(
    0,
    "AppointmentKey",
    range(
        1,
        len(fact_appointment) + 1
    )
)

In [132]:
fact_appointment[
    [
        "AppointmentKey",
        "AppointmentID",
        "PatientKey",
        "ProviderKey",
        "DepartmentKey",
        "ScheduledDateKey",
        "AppointmentDateKey"
    ]
].head(10)

,AppointmentKey,AppointmentID,PatientKey,ProviderKey,DepartmentKey,ScheduledDateKey,AppointmentDateKey
0,1,APT000001,1936,23,25,20251111,20251123
1,2,APT000002,9791,74,4,20250815,20250829
2,3,APT000003,8766,57,21,20250415,20250610
3,4,APT000004,4486,106,19,20240903,20241101
4,5,APT000005,9120,116,9,20250601,20250604
5,6,APT000006,4082,108,18,20241114,20241120
6,7,APT000007,2704,2,22,20251004,20251114
7,8,APT000008,4119,47,6,20250517,20250613
8,9,APT000009,9356,8,22,20241107,20241203
9,10,APT000010,7292,43,20,20250905,20250911


### Build final FactAppointment

In [133]:
fact_appointment_columns = [
    "AppointmentKey",
    "AppointmentID",

    "PatientKey",
    "ProviderKey",
    "DepartmentKey",

    "ScheduledDateKey",
    "AppointmentDateKey",

    "ScheduledDate",
    "AppointmentDateTime",

    "AppointmentStatus",
    "AppointmentType",
    "CancellationReason",

    "BookingLeadDays"
]

In [134]:
quality_columns = [
    "PatientIDValidFlag",
    "ProviderIDValidFlag",
    "DepartmentIDValidFlag",
    "CancellationReasonValidFlag",
    "StatusKPIEligibleFlag",
    "BookingLeadTimeValidFlag",
    "BookingLeadKPIEligibleFlag",
    "AppointmentQualityFlag"
]

quality_columns = [
    column
    for column in quality_columns
    if column in fact_appointment.columns
]

In [135]:
fact_appointment = fact_appointment[
    fact_appointment_columns
    + quality_columns
].copy()

In [136]:
fact_appointment.head()

,AppointmentKey,AppointmentID,PatientKey,ProviderKey,DepartmentKey,ScheduledDateKey,AppointmentDateKey,ScheduledDate,AppointmentDateTime,AppointmentStatus,AppointmentType,CancellationReason,BookingLeadDays,PatientIDValidFlag,ProviderIDValidFlag,DepartmentIDValidFlag,CancellationReasonValidFlag,BookingLeadTimeValidFlag
0,1,APT000001,1936,23,25,20251111,20251123,2025-11-11,2025-11-23 16:00:00,Completed,Post-Discharge Follow-Up,NaN,12.0,1,1,1,1,1
1,2,APT000002,9791,74,4,20250815,20250829,2025-08-15,2025-08-29 16:45:00,Completed,Routine,NaN,14.0,1,1,1,1,1
2,3,APT000003,8766,57,21,20250415,20250610,2025-04-15,2025-06-10 14:00:00,Completed,Specialist,NaN,56.0,1,1,1,1,1
3,4,APT000004,4486,106,19,20240903,20241101,2024-09-03,2024-11-01 16:15:00,Cancelled,Specialist,Scheduling Conflict,59.0,1,1,1,1,1
4,5,APT000005,9120,116,9,20250601,20250604,2025-06-01,2025-06-04 13:45:00,Completed,Follow-Up,NaN,3.0,1,1,1,1,1


### Validate the grain

In [137]:
print(
    "FactAppointment rows:",
    len(fact_appointment)
)

print(
    "Unique AppointmentIDs:",
    fact_appointment[
        "AppointmentID"
    ].nunique()
)

print(
    "Duplicate AppointmentID:",
    fact_appointment[
        "AppointmentID"
    ].duplicated().sum()
)

print(
    "Duplicate AppointmentKey:",
    fact_appointment[
        "AppointmentKey"
    ].duplicated().sum()
)

FactAppointment rows: 50000
Unique AppointmentIDs: 50000
Duplicate AppointmentID: 0
Duplicate AppointmentKey: 0


### Validate dimension keys

In [138]:
print(
    "Missing PatientKey:",
    fact_appointment[
        "PatientKey"
    ].isna().sum()
)

print(
    "Missing ProviderKey:",
    fact_appointment[
        "ProviderKey"
    ].isna().sum()
)

print(
    "Missing DepartmentKey:",
    fact_appointment[
        "DepartmentKey"
    ].isna().sum()
)

Missing PatientKey: 0
Missing ProviderKey: 0
Missing DepartmentKey: 0


### Validate date keys

In [139]:
for column in [
    "ScheduledDateKey",
    "AppointmentDateKey"
]:

    invalid_count = (
        ~fact_appointment[column]
        .isin(dim_date["DateKey"])
    ).sum()

    print(
        column,
        "invalid keys:",
        invalid_count
    )

ScheduledDateKey invalid keys: 0
AppointmentDateKey invalid keys: 0


### Validate BookingLeadDays

In [140]:
print(
    "Negative BookingLeadDays:",
    (
        fact_appointment[
            "BookingLeadDays"
        ] < 0
    ).sum()
)

Negative BookingLeadDays: 0


### Check appointment statuses

In [141]:
fact_appointment[
    "AppointmentStatus"
].value_counts(
    dropna=False
)

AppointmentStatus
Completed      37054
Cancelled       5020
No Show         4895
Rescheduled     2984
Unknown           47
Name: count, dtype: int64

In [142]:
fact_appointment.to_csv(
    WAREHOUSE_DATA_PATH
    / "fact_appointment.csv",
    index=False
)

print(
    "fact_appointment.csv created successfully."
)

fact_appointment.csv created successfully.


### Build FactEncounterDiagnosis

In [143]:
stg_encounter_diagnoses = pd.read_csv(
    STAGING_DATA_PATH
    / "stg_encounter_diagnoses.csv"
)

print(
    "Staging encounter diagnoses:",
    len(stg_encounter_diagnoses)
)

stg_encounter_diagnoses.head()

Staging encounter diagnoses: 153137


,EncounterDiagnosisID,EncounterID,DiagnosisID,DiagnosisType,PresentOnAdmissionFlag,DiagnosisIDValidFlag,SourceDiagnosisID
0,EDX0000001,ENC000001,DX026,Primary,NaN,1,DX026
1,EDX0000002,ENC000002,DX005,Primary,NaN,1,DX005
2,EDX0000003,ENC000003,DX044,Primary,NaN,1,DX044
3,EDX0000004,ENC000003,DX029,Secondary,NaN,1,DX029
4,EDX0000005,ENC000003,DX047,Secondary,NaN,1,DX047


In [144]:
stg_encounter_diagnoses.columns.tolist()

['EncounterDiagnosisID',
 'EncounterID',
 'DiagnosisID',
 'DiagnosisType',
 'PresentOnAdmissionFlag',
 'DiagnosisIDValidFlag',
 'SourceDiagnosisID']

### Start the fact table

In [145]:
fact_encounter_diagnosis = (
    stg_encounter_diagnoses.copy()
)

### Convert EncounterID → EncounterKey

In [146]:
encounter_key_map = (
    fact_encounter
    .set_index("EncounterID")[
        "EncounterKey"
    ]
)

In [147]:
fact_encounter_diagnosis[
    "EncounterKey"
] = (
    fact_encounter_diagnosis[
        "EncounterID"
    ]
    .map(encounter_key_map)
)

In [148]:
print(
    "Missing EncounterKey:",
    fact_encounter_diagnosis[
        "EncounterKey"
    ].isna().sum()
)

Missing EncounterKey: 0


### Convert DiagnosisID → DiagnosisKey

In [149]:
diagnosis_key_map = (
    dim_diagnosis
    .set_index("DiagnosisID")[
        "DiagnosisKey"
    ]
)

In [150]:
fact_encounter_diagnosis[
    "DiagnosisKey"
] = (
    fact_encounter_diagnosis[
        "DiagnosisID"
    ]
    .map(diagnosis_key_map)
    .fillna(0)
    .astype(int)
)

### Create EncounterDiagnosisKey

In [151]:
fact_encounter_diagnosis = (
    fact_encounter_diagnosis
    .sort_values("EncounterDiagnosisID")
    .reset_index(drop=True)
)

fact_encounter_diagnosis.insert(
    0,
    "EncounterDiagnosisKey",
    range(
        1,
        len(fact_encounter_diagnosis) + 1
    )
)

In [152]:
fact_encounter_diagnosis[
    [
        "EncounterDiagnosisKey",
        "EncounterDiagnosisID",
        "EncounterKey",
        "DiagnosisKey",
        "DiagnosisType",
        "PresentOnAdmissionFlag"
    ]
].head(10)

,EncounterDiagnosisKey,EncounterDiagnosisID,EncounterKey,DiagnosisKey,DiagnosisType,PresentOnAdmissionFlag
0,1,EDX0000001,1,26,Primary,NaN
1,2,EDX0000002,2,5,Primary,NaN
2,3,EDX0000003,3,44,Primary,NaN
3,4,EDX0000004,3,29,Secondary,NaN
4,5,EDX0000005,3,47,Secondary,NaN
5,6,EDX0000006,4,48,Primary,NaN
6,7,EDX0000007,4,28,Secondary,NaN
7,8,EDX0000008,4,30,Secondary,NaN
8,9,EDX0000009,5,15,Primary,NaN
9,10,EDX0000010,5,1,Secondary,NaN


In [153]:
fact_encounter_diagnosis_columns = [
    "EncounterDiagnosisKey",
    "EncounterDiagnosisID",

    "EncounterKey",
    "DiagnosisKey",

    "DiagnosisType",
    "PresentOnAdmissionFlag"
]

In [154]:
quality_columns = [
    "DiagnosisIDValidFlag"
]

quality_columns = [
    column
    for column in quality_columns
    if column
    in fact_encounter_diagnosis.columns
]

In [155]:
fact_encounter_diagnosis = (
    fact_encounter_diagnosis[
        fact_encounter_diagnosis_columns
        + quality_columns
    ]
    .copy()
)

In [156]:
fact_encounter_diagnosis.head()

,EncounterDiagnosisKey,EncounterDiagnosisID,EncounterKey,DiagnosisKey,DiagnosisType,PresentOnAdmissionFlag,DiagnosisIDValidFlag
0,1,EDX0000001,1,26,Primary,NaN,1
1,2,EDX0000002,2,5,Primary,NaN,1
2,3,EDX0000003,3,44,Primary,NaN,1
3,4,EDX0000004,3,29,Secondary,NaN,1
4,5,EDX0000005,3,47,Secondary,NaN,1


### Validate the primary key

In [157]:
print(
    "Rows:",
    len(fact_encounter_diagnosis)
)

print(
    "Duplicate EncounterDiagnosisKey:",
    fact_encounter_diagnosis[
        "EncounterDiagnosisKey"
    ].duplicated().sum()
)

print(
    "Duplicate EncounterDiagnosisID:",
    fact_encounter_diagnosis[
        "EncounterDiagnosisID"
    ].duplicated().sum()
)

Rows: 153137
Duplicate EncounterDiagnosisKey: 0
Duplicate EncounterDiagnosisID: 0


### Validate relationships

In [158]:
print(
    "Missing EncounterKey:",
    fact_encounter_diagnosis[
        "EncounterKey"
    ].isna().sum()
)

print(
    "Missing DiagnosisKey:",
    fact_encounter_diagnosis[
        "DiagnosisKey"
    ].isna().sum()
)

Missing EncounterKey: 0
Missing DiagnosisKey: 0


In [159]:
print(
    "Invalid EncounterKey:",
    (
        ~fact_encounter_diagnosis[
            "EncounterKey"
        ].isin(
            fact_encounter[
                "EncounterKey"
            ]
        )
    ).sum()
)

print(
    "Invalid DiagnosisKey:",
    (
        ~fact_encounter_diagnosis[
            "DiagnosisKey"
        ].isin(
            dim_diagnosis[
                "DiagnosisKey"
            ]
        )
    ).sum()
)

Invalid EncounterKey: 0
Invalid DiagnosisKey: 0


### Validate duplicate encounter-diagnosis combinations

In [160]:
duplicate_pairs = (
    fact_encounter_diagnosis
    .duplicated(
        subset=[
            "EncounterKey",
            "DiagnosisKey"
        ]
    )
    .sum()
)

print(
    "Duplicate Encounter-Diagnosis pairs:",
    duplicate_pairs
)

Duplicate Encounter-Diagnosis pairs: 1


In [161]:
duplicate_encounter_diagnosis_rows = (
    fact_encounter_diagnosis[
        fact_encounter_diagnosis.duplicated(
            subset=[
                "EncounterKey",
                "DiagnosisKey"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "EncounterKey",
            "DiagnosisKey"
        ]
    )
)

duplicate_encounter_diagnosis_rows

,EncounterDiagnosisKey,EncounterDiagnosisID,EncounterKey,DiagnosisKey,DiagnosisType,PresentOnAdmissionFlag,DiagnosisIDValidFlag
42201,42202,EDX0042202,24793,0,Primary,NaN,0
42202,42203,EDX0042203,24793,0,Secondary,NaN,0


In [162]:
duplicate_encounter_diagnosis_rows[
    [
        "EncounterDiagnosisID",
        "EncounterKey",
        "DiagnosisKey",
        "DiagnosisType",
        "PresentOnAdmissionFlag"
    ]
]

,EncounterDiagnosisID,EncounterKey,DiagnosisKey,DiagnosisType,PresentOnAdmissionFlag
42201,EDX0042202,24793,0,Primary,NaN
42202,EDX0042203,24793,0,Secondary,NaN


In [163]:
duplicate_source_check = (
    duplicate_encounter_diagnosis_rows[
        [
            "EncounterDiagnosisID",
            "EncounterKey",
            "DiagnosisKey"
        ]
    ]
    .merge(
        stg_encounter_diagnoses[
            [
                "EncounterDiagnosisID",
                "EncounterID",
                "DiagnosisID",
                "SourceDiagnosisID",
                "DiagnosisType"
            ]
        ],
        on="EncounterDiagnosisID",
        how="left"
    )
)

duplicate_source_check

,EncounterDiagnosisID,EncounterKey,DiagnosisKey,EncounterID,DiagnosisID,SourceDiagnosisID,DiagnosisType
0,EDX0042202,24793,0,ENC024793,UNKNOWN,DX999,Primary
1,EDX0042203,24793,0,ENC024793,UNKNOWN,DX999,Secondary


In [164]:
known_diagnosis_rows = (
    fact_encounter_diagnosis[
        fact_encounter_diagnosis[
            "DiagnosisKey"
        ] != 0
    ]
)

duplicate_known_pairs = (
    known_diagnosis_rows
    .duplicated(
        subset=[
            "EncounterKey",
            "DiagnosisKey"
        ]
    )
    .sum()
)

print(
    "Duplicate known Encounter-Diagnosis pairs:",
    duplicate_known_pairs
)

Duplicate known Encounter-Diagnosis pairs: 0


In [165]:
print(
    "UNKNOWN diagnosis assignments:",
    (
        fact_encounter_diagnosis[
            "DiagnosisKey"
        ] == 0
    ).sum()
)

UNKNOWN diagnosis assignments: 459


In [166]:
known_diagnosis_rows = (
    fact_encounter_diagnosis[
        fact_encounter_diagnosis["DiagnosisKey"] != 0
    ]
)

duplicate_known_pairs = (
    known_diagnosis_rows
    .duplicated(
        subset=[
            "EncounterKey",
            "DiagnosisKey"
        ]
    )
    .sum()
)

print(
    "Duplicate known Encounter-Diagnosis pairs:",
    duplicate_known_pairs
)

Duplicate known Encounter-Diagnosis pairs: 0


In [167]:
fact_encounter_diagnosis.to_csv(
    WAREHOUSE_DATA_PATH
    / "fact_encounter_diagnosis.csv",
    index=False
)

print(
    "fact_encounter_diagnosis.csv created successfully."
)

fact_encounter_diagnosis.csv created successfully.


### Build FactEncounterProcedure

In [168]:
stg_encounter_procedures = pd.read_csv(
    STAGING_DATA_PATH
    / "stg_encounter_procedures.csv"
)

print(
    "Staging encounter procedures:",
    len(stg_encounter_procedures)
)

stg_encounter_procedures.head()

Staging encounter procedures: 70419


,EncounterProcedureID,EncounterID,ProcedureID,ProcedureDate,ProcedureCost,ProcedureIDValidFlag,ProcedureCostValidFlag,ProcedureDateValidFlag,PotentialDuplicateFlag,SourceProcedureID,SourceProcedureCost
0,EPR0000001,ENC000003,PROC010,2025-10-02,1585.99,1,1,1,0,PROC010,1585.99
1,EPR0000002,ENC000003,PROC020,2025-10-01,128.26,1,1,1,0,PROC020,128.26
2,EPR0000003,ENC000005,PROC002,2025-12-16,93.15,1,1,1,0,PROC002,93.15
3,EPR0000004,ENC000007,PROC037,2024-07-23,43.40,1,1,1,0,PROC037,43.40
4,EPR0000005,ENC000007,PROC002,2024-07-23,99.74,1,1,1,0,PROC002,99.74


In [169]:
stg_encounter_procedures.columns.tolist()

['EncounterProcedureID',
 'EncounterID',
 'ProcedureID',
 'ProcedureDate',
 'ProcedureCost',
 'ProcedureIDValidFlag',
 'ProcedureCostValidFlag',
 'ProcedureDateValidFlag',
 'PotentialDuplicateFlag',
 'SourceProcedureID',
 'SourceProcedureCost']

### Start the fact table

In [170]:
fact_encounter_procedure = (
    stg_encounter_procedures.copy()
)

### EncounterID → EncounterKey

In [171]:
encounter_key_map = (
    fact_encounter
    .set_index("EncounterID")[
        "EncounterKey"
    ]
)

fact_encounter_procedure[
    "EncounterKey"
] = (
    fact_encounter_procedure[
        "EncounterID"
    ]
    .map(encounter_key_map)
)

In [172]:
print(
    "Missing EncounterKey:",
    fact_encounter_procedure[
        "EncounterKey"
    ].isna().sum()
)

Missing EncounterKey: 0


### ProcedureID → ProcedureKey

In [173]:
procedure_key_map = (
    dim_procedure
    .set_index("ProcedureID")[
        "ProcedureKey"
    ]
)

fact_encounter_procedure[
    "ProcedureKey"
] = (
    fact_encounter_procedure[
        "ProcedureID"
    ]
    .map(procedure_key_map)
    .fillna(0)
    .astype(int)
)

### Convert ProcedureDate

In [174]:
fact_encounter_procedure[
    "ProcedureDate"
] = pd.to_datetime(
    fact_encounter_procedure[
        "ProcedureDate"
    ],
    format="mixed",
    errors="coerce"
)

In [175]:
fact_encounter_procedure[
    "ProcedureDateKey"
] = (
    pd.to_numeric(
        fact_encounter_procedure[
            "ProcedureDate"
        ].dt.strftime("%Y%m%d"),
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

### Create EncounterProcedureKey

In [176]:
fact_encounter_procedure = (
    fact_encounter_procedure
    .sort_values(
        "EncounterProcedureID"
    )
    .reset_index(drop=True)
)

fact_encounter_procedure.insert(
    0,
    "EncounterProcedureKey",
    range(
        1,
        len(fact_encounter_procedure) + 1
    )
)

In [177]:
fact_encounter_procedure[
    [
        "EncounterProcedureKey",
        "EncounterProcedureID",
        "EncounterKey",
        "ProcedureKey",
        "ProcedureDateKey",
        "ProcedureCost"
    ]
].head(10)

,EncounterProcedureKey,EncounterProcedureID,EncounterKey,ProcedureKey,ProcedureDateKey,ProcedureCost
0,1,EPR0000001,3,10,20251002,1585.99
1,2,EPR0000002,3,20,20251001,128.26
2,3,EPR0000003,5,2,20251216,93.15
3,4,EPR0000004,7,37,20240723,43.40
4,5,EPR0000005,7,2,20240723,99.74
5,6,EPR0000006,8,1,20230709,75.43
6,7,EPR0000007,10,5,20241018,58.39
7,8,EPR0000008,11,1,20241109,87.93
8,9,EPR0000009,12,9,20250621,895.24
9,10,EPR0000010,13,40,20251009,273.96


In [178]:
fact_encounter_procedure_columns = [
    "EncounterProcedureKey",
    "EncounterProcedureID",

    "EncounterKey",
    "ProcedureKey",
    "ProcedureDateKey",

    "ProcedureDate",
    "ProcedureCost"
]

In [179]:
quality_columns = [
    "ProcedureIDValidFlag",
    "ProcedureCostValidFlag",
    "PotentialDuplicateFlag"
]

quality_columns = [
    column
    for column in quality_columns
    if column
    in fact_encounter_procedure.columns
]

In [180]:
fact_encounter_procedure = (
    fact_encounter_procedure[
        fact_encounter_procedure_columns
        + quality_columns
    ]
    .copy()
)

In [181]:
fact_encounter_procedure.head()

,EncounterProcedureKey,EncounterProcedureID,EncounterKey,ProcedureKey,ProcedureDateKey,ProcedureDate,ProcedureCost,ProcedureIDValidFlag,ProcedureCostValidFlag,PotentialDuplicateFlag
0,1,EPR0000001,3,10,20251002,2025-10-02,1585.99,1,1,0
1,2,EPR0000002,3,20,20251001,2025-10-01,128.26,1,1,0
2,3,EPR0000003,5,2,20251216,2025-12-16,93.15,1,1,0
3,4,EPR0000004,7,37,20240723,2024-07-23,43.40,1,1,0
4,5,EPR0000005,7,2,20240723,2024-07-23,99.74,1,1,0


### Validate grain

In [182]:
print(
    "Rows:",
    len(fact_encounter_procedure)
)

print(
    "Unique EncounterProcedureIDs:",
    fact_encounter_procedure[
        "EncounterProcedureID"
    ].nunique()
)

print(
    "Duplicate EncounterProcedureID:",
    fact_encounter_procedure[
        "EncounterProcedureID"
    ].duplicated().sum()
)

print(
    "Duplicate EncounterProcedureKey:",
    fact_encounter_procedure[
        "EncounterProcedureKey"
    ].duplicated().sum()
)

Rows: 70419
Unique EncounterProcedureIDs: 70419
Duplicate EncounterProcedureID: 0
Duplicate EncounterProcedureKey: 0


### Validate relationships

In [183]:
print(
    "Missing EncounterKey:",
    fact_encounter_procedure[
        "EncounterKey"
    ].isna().sum()
)

print(
    "Missing ProcedureKey:",
    fact_encounter_procedure[
        "ProcedureKey"
    ].isna().sum()
)

Missing EncounterKey: 0
Missing ProcedureKey: 0


In [184]:
print(
    "Invalid EncounterKey:",
    (
        ~fact_encounter_procedure[
            "EncounterKey"
        ].isin(
            fact_encounter[
                "EncounterKey"
            ]
        )
    ).sum()
)

print(
    "Invalid ProcedureKey:",
    (
        ~fact_encounter_procedure[
            "ProcedureKey"
        ].isin(
            dim_procedure[
                "ProcedureKey"
            ]
        )
    ).sum()
)

Invalid EncounterKey: 0
Invalid ProcedureKey: 0


### Validate ProcedureDateKey

In [185]:
print(
    "Invalid ProcedureDateKey:",
    (
        ~fact_encounter_procedure[
            "ProcedureDateKey"
        ].isin(
            dim_date[
                "DateKey"
            ]
        )
    ).sum()
)

Invalid ProcedureDateKey: 0


### Validate negative procedure cost

In [186]:
print(
    "Negative ProcedureCost:",
    (
        fact_encounter_procedure[
            "ProcedureCost"
        ] < 0
    ).sum()
)

Negative ProcedureCost: 0


### Check UNKNOWN procedure assignments

In [187]:
print(
    "UNKNOWN procedure assignments:",
    (
        fact_encounter_procedure[
            "ProcedureKey"
        ] == 0
    ).sum()
)

UNKNOWN procedure assignments: 140


### Validate duplicate known procedure combinations carefully

In [188]:
known_procedure_rows = (
    fact_encounter_procedure[
        fact_encounter_procedure[
            "ProcedureKey"
        ] != 0
    ]
)

repeated_known_pairs = (
    known_procedure_rows
    .duplicated(
        subset=[
            "EncounterKey",
            "ProcedureKey"
        ]
    )
    .sum()
)

print(
    "Repeated known Encounter-Procedure pairs:",
    repeated_known_pairs
)

Repeated known Encounter-Procedure pairs: 0


In [189]:
fact_encounter_procedure.to_csv(
    WAREHOUSE_DATA_PATH
    / "fact_encounter_procedure.csv",
    index=False
)

print(
    "fact_encounter_procedure.csv "
    "created successfully."
)

fact_encounter_procedure.csv created successfully.


### Build FactLabResult

In [190]:
stg_lab_results = pd.read_csv(
    STAGING_DATA_PATH
    / "stg_lab_results.csv"
)

print(
    "Staging lab result rows:",
    len(stg_lab_results)
)

stg_lab_results.head()

Staging lab result rows: 100000


,LabResultID,EncounterID,PatientID,LabTest,ResultValue,ResultUnit,ReferenceLow,ReferenceHigh,ResultFlag,ResultDateTime,...,ExpectedReferenceLow,ExpectedReferenceHigh,StatisticalOutlierFlag,ResultDateTimingStatus,ResultDateValidFlag,ResultDateLimitedConfidenceFlag,LabKPIEligibleFlag,SourcePatientID,SourceResultValue,SourceResultFlag
0,LAB0000001,ENC029144,PAT000376,LDL Cholesterol,137.56,mg/dL,0.0,100.0,High,2024-09-04 16:23:00,...,0.0,100.0,0,Valid,1,0,1,PAT000376,137.56,High
1,LAB0000002,ENC072776,PAT003049,Glucose,95.48,mg/dL,70.0,99.0,Normal,2024-02-28 09:47:00,...,70.0,99.0,0,Valid,1,0,1,PAT003049,95.48,Normal
2,LAB0000003,ENC006581,PAT009138,Hemoglobin,14.46,g/dL,12.0,17.5,Normal,2024-03-11 08:04:00,...,12.0,17.5,0,Valid,1,0,1,PAT009138,14.46,Normal
3,LAB0000004,ENC089691,PAT002228,Glucose,85.53,mg/dL,70.0,99.0,Normal,2025-08-01 10:18:00,...,70.0,99.0,0,Valid,1,0,1,PAT002228,85.53,Normal
4,LAB0000005,ENC015337,PAT008820,LDL Cholesterol,115.83,mg/dL,0.0,100.0,High,2025-10-12 14:09:00,...,0.0,100.0,0,Valid,1,0,1,PAT008820,115.83,High


In [191]:
stg_lab_results.columns.tolist()

['LabResultID',
 'EncounterID',
 'PatientID',
 'LabTest',
 'ResultValue',
 'ResultUnit',
 'ReferenceLow',
 'ReferenceHigh',
 'ResultFlag',
 'ResultDateTime',
 'EncounterIDValidFlag',
 'PatientIDValidFlag',
 'ResultUnitSource',
 'ExpectedResultUnit',
 'ReferenceLowSource',
 'ReferenceHighSource',
 'ExpectedReferenceLow',
 'ExpectedReferenceHigh',
 'StatisticalOutlierFlag',
 'ResultDateTimingStatus',
 'ResultDateValidFlag',
 'ResultDateLimitedConfidenceFlag',
 'LabKPIEligibleFlag',
 'SourcePatientID',
 'SourceResultValue',
 'SourceResultFlag']

In [192]:
fact_lab_result = stg_lab_results.copy()

In [193]:
encounter_key_map = (
    fact_encounter
    .set_index("EncounterID")[
        "EncounterKey"
    ]
)

In [194]:
fact_lab_result[
    "EncounterKey"
] = (
    fact_lab_result[
        "EncounterID"
    ]
    .map(encounter_key_map)
)

In [195]:
print(
    "Missing EncounterKey:",
    fact_lab_result[
        "EncounterKey"
    ].isna().sum()
)

Missing EncounterKey: 0


In [196]:
patient_key_map = (
    dim_patient
    .set_index("PatientID")[
        "PatientKey"
    ]
)

fact_lab_result[
    "PatientKey"
] = (
    fact_lab_result[
        "PatientID"
    ]
    .map(patient_key_map)
    .fillna(0)
    .astype(int)
)

In [197]:
fact_lab_result[
    "ResultDateTime"
] = pd.to_datetime(
    fact_lab_result[
        "ResultDateTime"
    ],
    format="mixed",
    errors="coerce"
)

In [198]:
fact_lab_result[
    "ResultDateKey"
] = (
    pd.to_numeric(
        fact_lab_result[
            "ResultDateTime"
        ].dt.strftime("%Y%m%d"),
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

In [199]:
fact_lab_result = (
    fact_lab_result
    .sort_values("LabResultID")
    .reset_index(drop=True)
)

fact_lab_result.insert(
    0,
    "LabResultKey",
    range(
        1,
        len(fact_lab_result) + 1
    )
)

In [200]:
fact_lab_result[
    [
        "LabResultKey",
        "LabResultID",
        "EncounterKey",
        "PatientKey",
        "ResultDateKey",
        "LabTest",
        "ResultValue",
        "ResultUnit",
        "ResultFlag"
    ]
].head(10)

,LabResultKey,LabResultID,EncounterKey,PatientKey,ResultDateKey,LabTest,ResultValue,ResultUnit,ResultFlag
0,1,LAB0000001,29144,371,20240904,LDL Cholesterol,137.56,mg/dL,High
1,2,LAB0000002,72776,3043,20240228,Glucose,95.48,mg/dL,Normal
2,3,LAB0000003,6581,9120,20240311,Hemoglobin,14.46,g/dL,Normal
3,4,LAB0000004,89691,2222,20250801,Glucose,85.53,mg/dL,Normal
4,5,LAB0000005,15337,8802,20251012,LDL Cholesterol,115.83,mg/dL,High
5,6,LAB0000006,57201,7144,20250119,Creatinine,0.55,mg/dL,Low
6,7,LAB0000007,26624,223,20250222,Creatinine,0.73,mg/dL,Normal
7,8,LAB0000008,77992,8387,20231231,Creatinine,0.95,mg/dL,Normal
8,9,LAB0000009,15035,2959,20251226,Glucose,153.19,mg/dL,Critical
9,10,LAB0000010,54044,8224,20251108,HbA1c,6.22,%,High


In [201]:
fact_lab_result_columns = [
    "LabResultKey",
    "LabResultID",

    "EncounterKey",
    "PatientKey",
    "ResultDateKey",

    "LabTest",
    "ResultValue",
    "ResultUnit",
    "ReferenceLow",
    "ReferenceHigh",
    "ResultFlag",

    "ResultDateTime"
]

In [202]:
quality_columns = [
    "StatisticalOutlierFlag",
    "ResultDateValidFlag",
    "ResultDateLimitedConfidenceFlag",
    "ResultDateTimingStatus",
    "LabKPIEligibleFlag",
    "CleanValueKPIEligibleFlag",
    "LabQualityFlag"
]

quality_columns = [
    column
    for column in quality_columns
    if column in fact_lab_result.columns
]

In [203]:
fact_lab_result = fact_lab_result[
    fact_lab_result_columns
    + quality_columns
].copy()

In [204]:
fact_lab_result.head()

,LabResultKey,LabResultID,EncounterKey,PatientKey,ResultDateKey,LabTest,ResultValue,ResultUnit,ReferenceLow,ReferenceHigh,ResultFlag,ResultDateTime,StatisticalOutlierFlag,ResultDateValidFlag,ResultDateLimitedConfidenceFlag,ResultDateTimingStatus,LabKPIEligibleFlag
0,1,LAB0000001,29144,371,20240904,LDL Cholesterol,137.56,mg/dL,0.0,100.0,High,2024-09-04 16:23:00,0,1,0,Valid,1
1,2,LAB0000002,72776,3043,20240228,Glucose,95.48,mg/dL,70.0,99.0,Normal,2024-02-28 09:47:00,0,1,0,Valid,1
2,3,LAB0000003,6581,9120,20240311,Hemoglobin,14.46,g/dL,12.0,17.5,Normal,2024-03-11 08:04:00,0,1,0,Valid,1
3,4,LAB0000004,89691,2222,20250801,Glucose,85.53,mg/dL,70.0,99.0,Normal,2025-08-01 10:18:00,0,1,0,Valid,1
4,5,LAB0000005,15337,8802,20251012,LDL Cholesterol,115.83,mg/dL,0.0,100.0,High,2025-10-12 14:09:00,0,1,0,Valid,1


In [205]:
print(
    "FactLabResult rows:",
    len(fact_lab_result)
)

print(
    "Unique LabResultIDs:",
    fact_lab_result[
        "LabResultID"
    ].nunique()
)

print(
    "Duplicate LabResultID:",
    fact_lab_result[
        "LabResultID"
    ].duplicated().sum()
)

print(
    "Duplicate LabResultKey:",
    fact_lab_result[
        "LabResultKey"
    ].duplicated().sum()
)

FactLabResult rows: 100000
Unique LabResultIDs: 100000
Duplicate LabResultID: 0
Duplicate LabResultKey: 0


In [206]:
print(
    "Missing EncounterKey:",
    fact_lab_result[
        "EncounterKey"
    ].isna().sum()
)

print(
    "Missing PatientKey:",
    fact_lab_result[
        "PatientKey"
    ].isna().sum()
)

Missing EncounterKey: 0
Missing PatientKey: 0


In [207]:
print(
    "Invalid EncounterKey:",
    (
        ~fact_lab_result[
            "EncounterKey"
        ].isin(
            fact_encounter[
                "EncounterKey"
            ]
        )
    ).sum()
)

print(
    "Invalid PatientKey:",
    (
        ~fact_lab_result[
            "PatientKey"
        ].isin(
            dim_patient[
                "PatientKey"
            ]
        )
    ).sum()
)

Invalid EncounterKey: 0
Invalid PatientKey: 0


In [208]:
print(
    "Invalid ResultDateKey:",
    (
        ~fact_lab_result[
            "ResultDateKey"
        ].isin(
            dim_date[
                "DateKey"
            ]
        )
    ).sum()
)

Invalid ResultDateKey: 0


In [209]:
fact_lab_result[
    "LabTest"
].value_counts(
    dropna=False
)

LabTest
Glucose            20076
LDL Cholesterol    20058
HbA1c              20042
Creatinine         19957
Hemoglobin         19867
Name: count, dtype: int64

In [210]:
if "StatisticalOutlierFlag" in fact_lab_result.columns:

    print(
        "Statistical outlier records:",
        (
            fact_lab_result[
                "StatisticalOutlierFlag"
            ] == 1
        ).sum()
    )

Statistical outlier records: 300


In [211]:
if (
    "ResultDateLimitedConfidenceFlag"
    in fact_lab_result.columns
):

    print(
        "Limited-confidence result timestamps:",
        (
            fact_lab_result[
                "ResultDateLimitedConfidenceFlag"
            ] == 1
        ).sum()
    )

Limited-confidence result timestamps: 75


In [212]:
print(
    "Missing analytical ResultValue:",
    fact_lab_result[
        "ResultValue"
    ].isna().sum()
)

Missing analytical ResultValue: 300


In [213]:
fact_lab_result.to_csv(
    WAREHOUSE_DATA_PATH
    / "fact_lab_result.csv",
    index=False
)

print(
    "fact_lab_result.csv created successfully."
)

fact_lab_result.csv created successfully.


# Create the warehouse dictionary

In [214]:
warehouse_tables = {
    "dim_patient": dim_patient,
    "dim_department": dim_department,
    "dim_provider": dim_provider,
    "dim_payer": dim_payer,
    "dim_diagnosis": dim_diagnosis,
    "dim_procedure": dim_procedure,
    "dim_date": dim_date,

    "fact_encounter": fact_encounter,
    "fact_admission": fact_admission,
    "fact_appointment": fact_appointment,
    "fact_encounter_diagnosis": fact_encounter_diagnosis,
    "fact_encounter_procedure": fact_encounter_procedure,
    "fact_lab_result": fact_lab_result
}

print(
    "Total warehouse tables:",
    len(warehouse_tables)
)

Total warehouse tables: 13


### View warehouse inventory

In [215]:
warehouse_inventory = pd.DataFrame([
    {
        "Table": table_name,
        "Rows": len(df),
        "Columns": len(df.columns)
    }
    for table_name, df in warehouse_tables.items()
])

warehouse_inventory

,Table,Rows,Columns
0,dim_patient,9981,11
1,dim_department,26,7
2,dim_provider,121,8
3,dim_payer,7,4
4,dim_diagnosis,51,6
5,dim_procedure,41,6
6,dim_date,1132,12
7,fact_encounter,90000,26
8,fact_admission,17740,19
9,fact_appointment,50000,18


### Validate dimension primary keys

In [216]:
dimension_key_checks = {
    "DimPatient": (
        dim_patient,
        "PatientKey"
    ),
    "DimDepartment": (
        dim_department,
        "DepartmentKey"
    ),
    "DimProvider": (
        dim_provider,
        "ProviderKey"
    ),
    "DimPayer": (
        dim_payer,
        "PayerKey"
    ),
    "DimDiagnosis": (
        dim_diagnosis,
        "DiagnosisKey"
    ),
    "DimProcedure": (
        dim_procedure,
        "ProcedureKey"
    ),
    "DimDate": (
        dim_date,
        "DateKey"
    )
}

for table_name, (
    df,
    key_column
) in dimension_key_checks.items():

    duplicate_keys = (
        df[key_column]
        .duplicated()
        .sum()
    )

    missing_keys = (
        df[key_column]
        .isna()
        .sum()
    )

    print(
        f"{table_name}: "
        f"Duplicate Keys = {duplicate_keys}, "
        f"Missing Keys = {missing_keys}"
    )

DimPatient: Duplicate Keys = 0, Missing Keys = 0
DimDepartment: Duplicate Keys = 0, Missing Keys = 0
DimProvider: Duplicate Keys = 0, Missing Keys = 0
DimPayer: Duplicate Keys = 0, Missing Keys = 0
DimDiagnosis: Duplicate Keys = 0, Missing Keys = 0
DimProcedure: Duplicate Keys = 0, Missing Keys = 0
DimDate: Duplicate Keys = 0, Missing Keys = 0


### Validate dimension business IDs

In [217]:
dimension_business_keys = {
    "DimPatient": (
        dim_patient,
        "PatientID"
    ),
    "DimDepartment": (
        dim_department,
        "DepartmentID"
    ),
    "DimProvider": (
        dim_provider,
        "ProviderID"
    ),
    "DimPayer": (
        dim_payer,
        "PayerID"
    ),
    "DimDiagnosis": (
        dim_diagnosis,
        "DiagnosisID"
    ),
    "DimProcedure": (
        dim_procedure,
        "ProcedureID"
    )
}

for table_name, (
    df,
    id_column
) in dimension_business_keys.items():

    duplicates = (
        df[id_column]
        .duplicated()
        .sum()
    )

    print(
        f"{table_name} duplicate "
        f"{id_column}: {duplicates}"
    )

DimPatient duplicate PatientID: 0
DimDepartment duplicate DepartmentID: 0
DimProvider duplicate ProviderID: 0
DimPayer duplicate PayerID: 0
DimDiagnosis duplicate DiagnosisID: 0
DimProcedure duplicate ProcedureID: 0


### Validate UNKNOWN members

In [218]:
unknown_checks = {
    "Patient": (
        dim_patient,
        "PatientID",
        "PatientKey"
    ),
    "Department": (
        dim_department,
        "DepartmentID",
        "DepartmentKey"
    ),
    "Provider": (
        dim_provider,
        "ProviderID",
        "ProviderKey"
    ),
    "Payer": (
        dim_payer,
        "PayerID",
        "PayerKey"
    ),
    "Diagnosis": (
        dim_diagnosis,
        "DiagnosisID",
        "DiagnosisKey"
    ),
    "Procedure": (
        dim_procedure,
        "ProcedureID",
        "ProcedureKey"
    )
}

for name, (
    df,
    id_column,
    key_column
) in unknown_checks.items():

    unknown_keys = df.loc[
        df[id_column] == "UNKNOWN",
        key_column
    ].tolist()

    print(
        f"{name} UNKNOWN key:",
        unknown_keys
    )

Patient UNKNOWN key: [0]
Department UNKNOWN key: [0]
Provider UNKNOWN key: [0]
Payer UNKNOWN key: [0]
Diagnosis UNKNOWN key: [0]
Procedure UNKNOWN key: [0]


In [219]:
print(
    "Date UNKNOWN key:",
    dim_date.loc[
        dim_date["DateKey"] == 0,
        "DateKey"
    ].tolist()
)

Date UNKNOWN key: [0]


### Validate fact-table grains

In [220]:
fact_grain_checks = {
    "FactEncounter": (
        fact_encounter,
        "EncounterKey",
        "EncounterID"
    ),
    "FactAdmission": (
        fact_admission,
        "AdmissionKey",
        "AdmissionID"
    ),
    "FactAppointment": (
        fact_appointment,
        "AppointmentKey",
        "AppointmentID"
    ),
    "FactEncounterDiagnosis": (
        fact_encounter_diagnosis,
        "EncounterDiagnosisKey",
        "EncounterDiagnosisID"
    ),
    "FactEncounterProcedure": (
        fact_encounter_procedure,
        "EncounterProcedureKey",
        "EncounterProcedureID"
    ),
    "FactLabResult": (
        fact_lab_result,
        "LabResultKey",
        "LabResultID"
    )
}

for table_name, (
    df,
    warehouse_key,
    business_key
) in fact_grain_checks.items():

    print(
        "\n",
        table_name
    )

    print(
        "Rows:",
        len(df)
    )

    print(
        f"Duplicate {warehouse_key}:",
        df[
            warehouse_key
        ].duplicated().sum()
    )

    print(
        f"Duplicate {business_key}:",
        df[
            business_key
        ].duplicated().sum()
    )


 FactEncounter
Rows: 90000
Duplicate EncounterKey: 0
Duplicate EncounterID: 0

 FactAdmission
Rows: 17740
Duplicate AdmissionKey: 0
Duplicate AdmissionID: 0

 FactAppointment
Rows: 50000
Duplicate AppointmentKey: 0
Duplicate AppointmentID: 0

 FactEncounterDiagnosis
Rows: 153137
Duplicate EncounterDiagnosisKey: 0
Duplicate EncounterDiagnosisID: 0

 FactEncounterProcedure
Rows: 70419
Duplicate EncounterProcedureKey: 0
Duplicate EncounterProcedureID: 0

 FactLabResult
Rows: 100000
Duplicate LabResultKey: 0
Duplicate LabResultID: 0


### Validate FactEncounter relationships

In [221]:
print(
    "Invalid PatientKey:",
    (
        ~fact_encounter[
            "PatientKey"
        ].isin(
            dim_patient[
                "PatientKey"
            ]
        )
    ).sum()
)

print(
    "Invalid ProviderKey:",
    (
        ~fact_encounter[
            "ProviderKey"
        ].isin(
            dim_provider[
                "ProviderKey"
            ]
        )
    ).sum()
)

print(
    "Invalid DepartmentKey:",
    (
        ~fact_encounter[
            "DepartmentKey"
        ].isin(
            dim_department[
                "DepartmentKey"
            ]
        )
    ).sum()
)

print(
    "Invalid PayerKey:",
    (
        ~fact_encounter[
            "PayerKey"
        ].isin(
            dim_payer[
                "PayerKey"
            ]
        )
    ).sum()
)

print(
    "Invalid EncounterDateKey:",
    (
        ~fact_encounter[
            "EncounterDateKey"
        ].isin(
            dim_date[
                "DateKey"
            ]
        )
    ).sum()
)

Invalid PatientKey: 0
Invalid ProviderKey: 0
Invalid DepartmentKey: 0
Invalid PayerKey: 0
Invalid EncounterDateKey: 0


### Validate FactAdmission

In [222]:
print(
    "Invalid Admission EncounterKey:",
    (
        ~fact_admission[
            "EncounterKey"
        ].isin(
            fact_encounter[
                "EncounterKey"
            ]
        )
    ).sum()
)

print(
    "Invalid Admission PatientKey:",
    (
        ~fact_admission[
            "PatientKey"
        ].isin(
            dim_patient[
                "PatientKey"
            ]
        )
    ).sum()
)

print(
    "Invalid Admission DepartmentKey:",
    (
        ~fact_admission[
            "DepartmentKey"
        ].isin(
            dim_department[
                "DepartmentKey"
            ]
        )
    ).sum()
)

Invalid Admission EncounterKey: 0
Invalid Admission PatientKey: 0
Invalid Admission DepartmentKey: 0


In [223]:
for column in [
    "AdmissionDateKey",
    "DischargeDateKey",
    "FollowUpDateKey"
]:

    invalid = (
        ~fact_admission[
            column
        ].isin(
            dim_date[
                "DateKey"
            ]
        )
    ).sum()

    print(
        f"Invalid {column}:",
        invalid
    )

Invalid AdmissionDateKey: 0
Invalid DischargeDateKey: 0
Invalid FollowUpDateKey: 0


### Validate FactAppointment

In [224]:
print(
    "Invalid Appointment PatientKey:",
    (
        ~fact_appointment[
            "PatientKey"
        ].isin(
            dim_patient[
                "PatientKey"
            ]
        )
    ).sum()
)

print(
    "Invalid Appointment ProviderKey:",
    (
        ~fact_appointment[
            "ProviderKey"
        ].isin(
            dim_provider[
                "ProviderKey"
            ]
        )
    ).sum()
)

print(
    "Invalid Appointment DepartmentKey:",
    (
        ~fact_appointment[
            "DepartmentKey"
        ].isin(
            dim_department[
                "DepartmentKey"
            ]
        )
    ).sum()
)

Invalid Appointment PatientKey: 0
Invalid Appointment ProviderKey: 0
Invalid Appointment DepartmentKey: 0


In [225]:
for column in [
    "ScheduledDateKey",
    "AppointmentDateKey"
]:

    invalid = (
        ~fact_appointment[
            column
        ].isin(
            dim_date[
                "DateKey"
            ]
        )
    ).sum()

    print(
        f"Invalid {column}:",
        invalid
    )

Invalid ScheduledDateKey: 0
Invalid AppointmentDateKey: 0


### Validate Diagnosis, Procedure and Lab facts

In [226]:
print(
    "Invalid Diagnosis EncounterKey:",
    (
        ~fact_encounter_diagnosis[
            "EncounterKey"
        ].isin(
            fact_encounter[
                "EncounterKey"
            ]
        )
    ).sum()
)

print(
    "Invalid DiagnosisKey:",
    (
        ~fact_encounter_diagnosis[
            "DiagnosisKey"
        ].isin(
            dim_diagnosis[
                "DiagnosisKey"
            ]
        )
    ).sum()
)

Invalid Diagnosis EncounterKey: 0
Invalid DiagnosisKey: 0


In [227]:
print(
    "Invalid Procedure EncounterKey:",
    (
        ~fact_encounter_procedure[
            "EncounterKey"
        ].isin(
            fact_encounter[
                "EncounterKey"
            ]
        )
    ).sum()
)

print(
    "Invalid ProcedureKey:",
    (
        ~fact_encounter_procedure[
            "ProcedureKey"
        ].isin(
            dim_procedure[
                "ProcedureKey"
            ]
        )
    ).sum()
)

print(
    "Invalid ProcedureDateKey:",
    (
        ~fact_encounter_procedure[
            "ProcedureDateKey"
        ].isin(
            dim_date[
                "DateKey"
            ]
        )
    ).sum()
)

Invalid Procedure EncounterKey: 0
Invalid ProcedureKey: 0
Invalid ProcedureDateKey: 0


In [228]:
print(
    "Invalid Lab EncounterKey:",
    (
        ~fact_lab_result[
            "EncounterKey"
        ].isin(
            fact_encounter[
                "EncounterKey"
            ]
        )
    ).sum()
)

print(
    "Invalid Lab PatientKey:",
    (
        ~fact_lab_result[
            "PatientKey"
        ].isin(
            dim_patient[
                "PatientKey"
            ]
        )
    ).sum()
)

print(
    "Invalid ResultDateKey:",
    (
        ~fact_lab_result[
            "ResultDateKey"
        ].isin(
            dim_date[
                "DateKey"
            ]
        )
    ).sum()
)

Invalid Lab EncounterKey: 0
Invalid Lab PatientKey: 0
Invalid ResultDateKey: 0


### Final numerical sanity checks

In [229]:
negative_duration_rows = fact_encounter[
    fact_encounter["EncounterDurationHours"] < 0
].copy()

columns_to_show = [
    "EncounterID",
    "ArrivalDateTime",
    "EncounterEndDateTime",
    "EncounterDurationHours"
]

for col in [
    "EncounterEndRecoveredFlag",
    "EncounterDurationValidFlag",
    "EncounterQualityFlag"
]:
    if col in negative_duration_rows.columns:
        columns_to_show.append(col)

negative_duration_rows[
    columns_to_show
]

,EncounterID,ArrivalDateTime,EncounterEndDateTime,EncounterDurationHours,EncounterEndRecoveredFlag,EncounterDurationValidFlag,EncounterQualityFlag


In [230]:
negative_duration_rows[
    "RecalculatedDurationHours"
] = (
    negative_duration_rows[
        "EncounterEndDateTime"
    ]
    -
    negative_duration_rows[
        "ArrivalDateTime"
    ]
).dt.total_seconds() / 3600

In [231]:
negative_duration_rows[
    [
        "EncounterID",
        "ArrivalDateTime",
        "EncounterEndDateTime",
        "EncounterDurationHours",
        "RecalculatedDurationHours"
    ]
]

,EncounterID,ArrivalDateTime,EncounterEndDateTime,EncounterDurationHours,RecalculatedDurationHours


In [232]:
print(
    "Stored negative durations:",
    (
        negative_duration_rows[
            "EncounterDurationHours"
        ] < 0
    ).sum()
)

print(
    "Recalculated negative durations:",
    (
        negative_duration_rows[
            "RecalculatedDurationHours"
        ] < 0
    ).sum()
)

Stored negative durations: 0
Recalculated negative durations: 0


In [233]:
if "EncounterEndRecoveredFlag" in negative_duration_rows.columns:

    print(
        negative_duration_rows[
            "EncounterEndRecoveredFlag"
        ].value_counts(
            dropna=False
        )
    )

Series([], Name: count, dtype: int64)


In [234]:
recovered_negative_mask = (
    (fact_encounter["EncounterEndRecoveredFlag"] == 1)
    &
    (fact_encounter["EncounterDurationHours"] < 0)
)

print(
    "Rows to correct:",
    recovered_negative_mask.sum()
)

Rows to correct: 0


In [235]:
fact_encounter.loc[
    recovered_negative_mask,
    "EncounterDurationHours"
] = (
    fact_encounter.loc[
        recovered_negative_mask,
        "EncounterEndDateTime"
    ]
    -
    fact_encounter.loc[
        recovered_negative_mask,
        "ArrivalDateTime"
    ]
).dt.total_seconds() / 3600

In [236]:
if "EncounterDurationValidFlag" in fact_encounter.columns:

    fact_encounter.loc[
        recovered_negative_mask,
        "EncounterDurationValidFlag"
    ] = 1

In [237]:
print(
    "Negative EncounterDurationHours after correction:",
    (
        fact_encounter[
            "EncounterDurationHours"
        ] < 0
    ).sum()
)

Negative EncounterDurationHours after correction: 0


In [238]:
fact_encounter.loc[
    recovered_negative_mask,
    [
        "EncounterID",
        "ArrivalDateTime",
        "EncounterEndDateTime",
        "EncounterDurationHours",
        "EncounterEndRecoveredFlag"
    ]
].head()

,EncounterID,ArrivalDateTime,EncounterEndDateTime,EncounterDurationHours,EncounterEndRecoveredFlag


In [239]:
fact_encounter.to_csv(
    WAREHOUSE_DATA_PATH
    / "fact_encounter.csv",
    index=False
)

print(
    "Corrected fact_encounter.csv saved."
)

Corrected fact_encounter.csv saved.


In [240]:
print(
    "Negative EncounterCost:",
    (
        fact_encounter[
            "EncounterCost"
        ] < 0
    ).sum()
)

print(
    "Negative WaitMinutes:",
    (
        fact_encounter[
            "WaitMinutes"
        ] < 0
    ).sum()
)

print(
    "Negative EncounterDurationHours:",
    (
        fact_encounter[
            "EncounterDurationHours"
        ] < 0
    ).sum()
)

print(
    "Negative LengthOfStayDays:",
    (
        fact_admission[
            "LengthOfStayDays"
        ] < 0
    ).sum()
)

print(
    "Negative BookingLeadDays:",
    (
        fact_appointment[
            "BookingLeadDays"
        ] < 0
    ).sum()
)

print(
    "Negative ProcedureCost:",
    (
        fact_encounter_procedure[
            "ProcedureCost"
        ] < 0
    ).sum()
)

Negative EncounterCost: 0
Negative WaitMinutes: 0
Negative EncounterDurationHours: 0
Negative LengthOfStayDays: 0
Negative BookingLeadDays: 0
Negative ProcedureCost: 0


In [241]:
duplicate_known_diagnosis_pairs = (
    fact_encounter_diagnosis[
        fact_encounter_diagnosis["DiagnosisKey"] != 0
    ]
    .duplicated(
        subset=[
            "EncounterKey",
            "DiagnosisKey"
        ]
    )
    .sum()
)

print(
    "Duplicate known Encounter-Diagnosis pairs:",
    duplicate_known_diagnosis_pairs
)

print(
    "UNKNOWN diagnosis assignments:",
    (
        fact_encounter_diagnosis["DiagnosisKey"] == 0
    ).sum()
)

Duplicate known Encounter-Diagnosis pairs: 0
UNKNOWN diagnosis assignments: 459


In [242]:
repeated_known_procedure_pairs = (
    fact_encounter_procedure[
        fact_encounter_procedure["ProcedureKey"] != 0
    ]
    .duplicated(
        subset=[
            "EncounterKey",
            "ProcedureKey"
        ]
    )
    .sum()
)

print(
    "Repeated known Encounter-Procedure pairs:",
    repeated_known_procedure_pairs
)

Repeated known Encounter-Procedure pairs: 0


In [243]:
warehouse_inventory.to_csv(
    WAREHOUSE_DATA_PATH
    / "warehouse_inventory.csv",
    index=False
)

print(
    "warehouse_inventory.csv saved."
)

warehouse_inventory.csv saved.


In [244]:
expected_warehouse_files = [
    "dim_patient.csv",
    "dim_department.csv",
    "dim_provider.csv",
    "dim_payer.csv",
    "dim_diagnosis.csv",
    "dim_procedure.csv",
    "dim_date.csv",

    "fact_encounter.csv",
    "fact_admission.csv",
    "fact_appointment.csv",
    "fact_encounter_diagnosis.csv",
    "fact_encounter_procedure.csv",
    "fact_lab_result.csv"
]

existing_count = 0

for filename in expected_warehouse_files:

    path = WAREHOUSE_DATA_PATH / filename

    exists = path.exists()

    if exists:
        existing_count += 1

    print(
        filename,
        "✅ EXISTS"
        if exists
        else "❌ MISSING"
    )

print(
    f"\nWarehouse files available: "
    f"{existing_count}/13"
)

dim_patient.csv ✅ EXISTS
dim_department.csv ✅ EXISTS
dim_provider.csv ✅ EXISTS
dim_payer.csv ✅ EXISTS
dim_diagnosis.csv ✅ EXISTS
dim_procedure.csv ✅ EXISTS
dim_date.csv ✅ EXISTS
fact_encounter.csv ✅ EXISTS
fact_admission.csv ✅ EXISTS
fact_appointment.csv ✅ EXISTS
fact_encounter_diagnosis.csv ✅ EXISTS
fact_encounter_procedure.csv ✅ EXISTS
fact_lab_result.csv ✅ EXISTS

Warehouse files available: 13/13


In [245]:
check_admission = pd.read_csv(
    WAREHOUSE_DATA_PATH / "fact_admission.csv"
)

check_encounter = pd.read_csv(
    WAREHOUSE_DATA_PATH / "fact_encounter.csv"
)

print(
    "Warehouse admissions LOS > 10 days:",
    (check_admission["LengthOfStayDays"] > 10).sum()
)

print(
    "Warehouse negative LOS:",
    (check_admission["LengthOfStayDays"] < 0).sum()
)

print(
    "Warehouse maximum LOS:",
    check_admission["LengthOfStayDays"].max()
)

print(
    "Warehouse negative encounter durations:",
    (check_encounter["EncounterDurationHours"] < 0).sum()
)

print(
    "Warehouse maximum encounter duration:",
    check_encounter["EncounterDurationHours"].max()
)

Warehouse admissions LOS > 10 days: 0
Warehouse negative LOS: 0
Warehouse maximum LOS: 7.121889936550926
Warehouse negative encounter durations: 0
Warehouse maximum encounter duration: 59.998333333333335
